# Task M — AUG Walk-Forward Sensitivity Analysis

This notebook evaluates the sensitivity of the AUG / SHFE Gold Futures walk-forward results to reasonable changes in the in-sample estimation window $T$ and the out-of-sample evaluation horizon $\tau$.

The baseline specification is the finalized Task K design:

- **4 years** of prior data as the in-sample optimization window
- **3 months** as the out-of-sample evaluation window
- full parameter search over `ChnLen = 500..10000` in steps of `10`
- full parameter search over `StpPct = 0.005..0.100` in steps of `0.001`
- objective function: `Net Profit / |Maximum Drawdown|`
- the same finalized Channel WithDDControl strategy engine and transaction-cost assumptions used in Task K

The sensitivity analysis varies only $T$ and $\tau$:

$$
T \in \{4,5,6\}\text{ years},
\qquad
\tau \in \{3,6\}\text{ months}.
$$

This produces six walk-forward specifications:

$$
(4,3),\ (4,6),\ (5,3),\ (5,6),\ (6,3),\ (6,6).
$$

The AUG dataset begins in May 2018 and ends in April 2026. Therefore, an 8-year in-sample window, which was feasible in the longer-history soybean sensitivity analysis, cannot generate any subsequent AUG out-of-sample observations. A 6-year maximum in-sample window is used instead so that each tested specification retains a genuine OOS evaluation period.

The purpose of this exercise is **not** to select a new ex-post optimal walk-forward specification. Instead, it tests whether the strong AUG OOS results from Task K are broadly robust to reasonable alternative choices of $T$ and $\tau$.

The original `(4, 3)` Task K specification remains the baseline throughout.

## 0. Imports, Paths, and Sensitivity Configuration

This section defines the AUG market constants, the professor-specified parameter grid, and the six \(T/\tau\) sensitivity specifications.

All strategy logic, transaction-cost assumptions, and optimization criteria are kept consistent with the finalized Task K framework. Only the walk-forward window design is varied.

In [1]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False

    def njit(func=None, **kwargs):
        return func if func is not None else (lambda f: f)


# ------------------------------------------------------------
# Display settings
# ------------------------------------------------------------

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:,.6f}".format)


# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        if (candidate / "data").exists():
            return candidate

    raise RuntimeError(
        "Could not locate repository root containing the data/ directory."
    )


REPO_ROOT = find_repo_root()

CLEAN_DATA_FILE = REPO_ROOT / "data" / "clean" / "AUG.parquet"
RAW_DATA_FILE = REPO_ROOT / "data" / "AUG-5minHLV.csv"

OUT_DIR = REPO_ROOT / "task_m_aug_sensitivity_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# AUG / SHFE Gold Futures configuration
# ------------------------------------------------------------

PV = 1000.0
SLPG = 0.065
E0 = 100_000.0

MARKET_NAME = "AUG / SHFE Gold Futures"
CURRENCY = "CNY"


# ------------------------------------------------------------
# Professor-specified full optimization grid
# ------------------------------------------------------------

CHNLEN_GRID = np.arange(
    500,
    10001,
    10,
    dtype=int
)

STPPCT_GRID = np.round(
    np.arange(
        0.005,
        0.1001,
        0.001
    ),
    3
)

N_GRID_COMBINATIONS = (
    len(CHNLEN_GRID)
    *
    len(STPPCT_GRID)
)


# ------------------------------------------------------------
# Walk-forward sensitivity specifications
#
# AUG has less than 8 complete years of history.
# Therefore T = 8 cannot produce a subsequent OOS period.
#
# We use T = {4, 5, 6} years instead.
#
# (4, 3) remains the finalized Task K baseline.
# ------------------------------------------------------------

SENSITIVITY_SPECS = [
    {
        "T_years": 4,
        "tau_months": 3,
        "label": "T4_tau3",
        "baseline": True,
    },
    {
        "T_years": 4,
        "tau_months": 6,
        "label": "T4_tau6",
        "baseline": False,
    },
    {
        "T_years": 5,
        "tau_months": 3,
        "label": "T5_tau3",
        "baseline": False,
    },
    {
        "T_years": 5,
        "tau_months": 6,
        "label": "T5_tau6",
        "baseline": False,
    },
    {
        "T_years": 6,
        "tau_months": 3,
        "label": "T6_tau3",
        "baseline": False,
    },
    {
        "T_years": 6,
        "tau_months": 6,
        "label": "T6_tau6",
        "baseline": False,
    },
]


# ------------------------------------------------------------
# Configuration audit
# ------------------------------------------------------------

print("=" * 72)
print("TASK M — AUG WALK-FORWARD SENSITIVITY ANALYSIS")
print("=" * 72)

print(f"Market               : {MARKET_NAME}")
print(f"Currency             : {CURRENCY}")
print(f"Point value          : {PV:,.2f}")
print(f"Round-turn slippage  : {SLPG:.3f}")
print(f"Initial equity       : {E0:,.2f}")

print()
print("Full optimization grid")

print(
    f"ChnLen               : "
    f"{CHNLEN_GRID[0]} to {CHNLEN_GRID[-1]} "
    f"(step {CHNLEN_GRID[1] - CHNLEN_GRID[0]})"
)

print(
    f"StpPct               : "
    f"{STPPCT_GRID[0]:.3f} to {STPPCT_GRID[-1]:.3f} "
    f"(step {STPPCT_GRID[1] - STPPCT_GRID[0]:.3f})"
)

print(
    f"Grid combinations    : "
    f"{N_GRID_COMBINATIONS:,}"
)

print()
print("Sensitivity specifications")

for spec in SENSITIVITY_SPECS:

    baseline_text = (
        "  <-- Task K baseline"
        if spec["baseline"]
        else ""
    )

    print(
        f"{spec['label']:10s}: "
        f"T = {spec['T_years']} years, "
        f"tau = {spec['tau_months']} months"
        f"{baseline_text}"
    )

print()
print(
    "Note: T = 8 years is infeasible for AUG because "
    "the available history ends before an 8-year IS window "
    "can be followed by a genuine OOS period."
)

print()
print(f"Numba available      : {NUMBA_AVAILABLE}")
print(f"Output directory     : {OUT_DIR}")

TASK M — AUG WALK-FORWARD SENSITIVITY ANALYSIS
Market               : AUG / SHFE Gold Futures
Currency             : CNY
Point value          : 1,000.00
Round-turn slippage  : 0.065
Initial equity       : 100,000.00

Full optimization grid
ChnLen               : 500 to 10000 (step 10)
StpPct               : 0.005 to 0.100 (step 0.001)
Grid combinations    : 91,296

Sensitivity specifications
T4_tau3   : T = 4 years, tau = 3 months  <-- Task K baseline
T4_tau6   : T = 4 years, tau = 6 months
T5_tau3   : T = 5 years, tau = 3 months
T5_tau6   : T = 5 years, tau = 6 months
T6_tau3   : T = 6 years, tau = 3 months
T6_tau6   : T = 6 years, tau = 6 months

Note: T = 8 years is infeasible for AUG because the available history ends before an 8-year IS window can be followed by a genuine OOS period.

Numba available      : True
Output directory     : /Users/apple/Desktop/期货市场趋势跟踪策略的量化分析与滚动回测/数据/task_m_aug_sensitivity_output


## 1. Load and Validate the Finalized AUG Dataset

This section loads the same finalized AUG dataset used in Task K and verifies its basic structure before any sensitivity analysis is performed.

The purpose is to ensure that differences across the six sensitivity specifications arise only from changes in \(T\) and \(\tau\), rather than from changes in the underlying market data.

In [2]:
if CLEAN_DATA_FILE.exists():
    aug = pd.read_parquet(CLEAN_DATA_FILE)
    data_source = CLEAN_DATA_FILE

elif RAW_DATA_FILE.exists():
    aug = pd.read_csv(RAW_DATA_FILE)
    data_source = RAW_DATA_FILE

else:
    raise FileNotFoundError(
        "Could not find either the finalized AUG parquet file "
        "or the raw AUG CSV file."
    )


# ------------------------------------------------------------
# Standardize timestamp index
# ------------------------------------------------------------

if isinstance(aug.index, pd.DatetimeIndex):
    aug.index = pd.to_datetime(aug.index)

else:
    datetime_candidates = [
        "ts",
        "DateTime",
        "Datetime",
        "datetime",
        "Timestamp",
        "timestamp",
        "Date",
        "date",
    ]

    datetime_col = None

    for col in datetime_candidates:
        if col in aug.columns:
            datetime_col = col
            break

    if datetime_col is None:
        raise ValueError(
            "No timestamp column was found and the index is not a DatetimeIndex."
        )

    aug[datetime_col] = pd.to_datetime(
        aug[datetime_col],
        errors="coerce"
    )

    if aug[datetime_col].isna().any():
        raise ValueError(
            f"Timestamp parsing failed for {aug[datetime_col].isna().sum()} rows."
        )

    aug = aug.set_index(datetime_col)


aug.index.name = "ts"
aug = aug.sort_index()


# ------------------------------------------------------------
# Standardize required OHLC column names
# ------------------------------------------------------------

column_map = {}

for col in aug.columns:
    lower = str(col).strip().lower()

    if lower == "open":
        column_map[col] = "Open"
    elif lower == "high":
        column_map[col] = "High"
    elif lower == "low":
        column_map[col] = "Low"
    elif lower == "close":
        column_map[col] = "Close"
    elif lower == "volume":
        column_map[col] = "Volume"

aug = aug.rename(columns=column_map)


required_cols = ["Open", "High", "Low", "Close"]

missing_cols = [
    col for col in required_cols
    if col not in aug.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required OHLC columns: {missing_cols}"
    )


# ------------------------------------------------------------
# Basic data-integrity checks
# ------------------------------------------------------------

n_rows = len(aug)

n_duplicate_timestamps = int(
    aug.index.duplicated().sum()
)

n_missing_ohlc = int(
    aug[required_cols].isna().sum().sum()
)

n_nonpositive_prices = int(
    (aug[required_cols] <= 0).sum().sum()
)

ohlc_order_violations = int(
    (
        (aug["High"] < aug[["Open", "Close", "Low"]].max(axis=1))
        |
        (aug["Low"] > aug[["Open", "Close", "High"]].min(axis=1))
    ).sum()
)


# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

print("=" * 72)
print("AUG DATA VALIDATION")
print("=" * 72)

print(f"Data source            : {data_source}")
print(f"Rows                   : {n_rows:,}")
print(f"Start timestamp        : {aug.index.min()}")
print(f"End timestamp          : {aug.index.max()}")

print()
print("Integrity checks")
print(f"Duplicate timestamps   : {n_duplicate_timestamps:,}")
print(f"Missing OHLC values    : {n_missing_ohlc:,}")
print(f"Non-positive prices    : {n_nonpositive_prices:,}")
print(f"OHLC ordering errors   : {ohlc_order_violations:,}")

print()
print("Columns")
print(list(aug.columns))

print()
print("Expected finalized Task K reference")
print("Rows                   : 138,744")
print("Start timestamp        : 2018-05-03 09:05:00")
print("End timestamp          : 2026-04-10 15:00:00")


# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

assert n_rows == 138_744, (
    f"Unexpected AUG row count: {n_rows:,}"
)

assert aug.index.min() == pd.Timestamp("2018-05-03 09:05:00")

assert aug.index.max() == pd.Timestamp("2026-04-10 15:00:00")

assert n_duplicate_timestamps == 0

assert n_missing_ohlc == 0

assert n_nonpositive_prices == 0

assert ohlc_order_violations == 0


print()
print("ALL FINALIZED AUG DATA CHECKS PASSED.")

AUG DATA VALIDATION
Data source            : /Users/apple/Desktop/期货市场趋势跟踪策略的量化分析与滚动回测/数据/data/clean/AUG.parquet
Rows                   : 138,744
Start timestamp        : 2018-05-03 09:05:00
End timestamp          : 2026-04-10 15:00:00

Integrity checks
Duplicate timestamps   : 0
Missing OHLC values    : 0
Non-positive prices    : 0
OHLC ordering errors   : 0

Columns
['Open', 'High', 'Low', 'Close', 'Volume', 'ret', 'logret']

Expected finalized Task K reference
Rows                   : 138,744
Start timestamp        : 2018-05-03 09:05:00
End timestamp          : 2026-04-10 15:00:00

ALL FINALIZED AUG DATA CHECKS PASSED.


## 2. Exact Strategy Engine and Optimization Functions

Task M reuses the same finalized Channel WithDDControl implementation and exact full-grid optimization framework validated in Task K.

No strategy logic is changed in this sensitivity analysis. In particular:

- channel levels use only prior bars
- the current bar is excluded from channel construction
- the same stop-loss and reversal logic is preserved
- the same AUG point value and slippage assumptions are used
- the full professor-specified parameter grid is evaluated
- the optimization objective remains `Net Profit / |Maximum Drawdown|`

Therefore, any differences across sensitivity specifications are attributable to the walk-forward window design rather than changes in the strategy engine.

In [3]:
HIGH = np.ascontiguousarray(
    aug["High"].to_numpy(dtype=np.float64)
)

LOW = np.ascontiguousarray(
    aug["Low"].to_numpy(dtype=np.float64)
)

CLOSE = np.ascontiguousarray(
    aug["Close"].to_numpy(dtype=np.float64)
)

N = len(aug)


# ------------------------------------------------------------
# Numba parallel helper
# ------------------------------------------------------------

if NUMBA_AVAILABLE:
    from numba import prange
else:
    prange = range


# ============================================================
# 2.1 O(N) prior-bar rolling channels
#
# HH[k] = max(High[k-L : k])
# LL[k] = min(Low[k-L : k])
#
# Current bar k is excluded.
# ============================================================

@njit
def build_channels_deque(
    high,
    low,
    length
):

    n = len(high)

    hh = np.empty(
        n,
        dtype=np.float64
    )

    ll = np.empty(
        n,
        dtype=np.float64
    )

    # Same initialization convention as finalized Task K.
    hh[:] = np.inf
    ll[:] = -np.inf

    max_deque = np.empty(
        n,
        dtype=np.int64
    )

    min_deque = np.empty(
        n,
        dtype=np.int64
    )

    max_head = 0
    max_tail = 0

    min_head = 0
    min_tail = 0

    for k in range(n):

        # ----------------------------------------------------
        # Remove observations outside [k-length, k-1].
        # ----------------------------------------------------

        lower_bound = k - length

        while (
            max_head < max_tail
            and
            max_deque[max_head] < lower_bound
        ):
            max_head += 1

        while (
            min_head < min_tail
            and
            min_deque[min_head] < lower_bound
        ):
            min_head += 1

        # ----------------------------------------------------
        # Channel for bar k uses PRIOR bars only.
        # ----------------------------------------------------

        if k >= length:

            hh[k] = high[
                max_deque[max_head]
            ]

            ll[k] = low[
                min_deque[min_head]
            ]

        # ----------------------------------------------------
        # Add current bar AFTER computing HH[k] / LL[k].
        # Therefore current bar is excluded.
        # ----------------------------------------------------

        while (
            max_head < max_tail
            and
            high[max_deque[max_tail - 1]]
            <= high[k]
        ):
            max_tail -= 1

        max_deque[max_tail] = k
        max_tail += 1

        while (
            min_head < min_tail
            and
            low[min_deque[min_tail - 1]]
            >= low[k]
        ):
            min_tail -= 1

        min_deque[min_tail] = k
        min_tail += 1

    return hh, ll


# ============================================================
# 2.2 Exact finalized Task K strategy scorer
# ============================================================

@njit
def score_strategy_fast(
    high,
    low,
    close,
    hh,
    ll,
    length,
    stop_pct,
    slpg,
    pv,
    initial_equity
):

    n = len(close)

    equity = initial_equity
    equity_max = initial_equity
    min_dd = 0.0

    position = 0

    benchmark_long = 0.0
    benchmark_short = 0.0

    for k in range(
        length,
        n
    ):

        traded = False

        # Existing-position mark-to-market P&L.
        delta = (
            pv
            *
            (close[k] - close[k - 1])
            *
            position
        )

        # ====================================================
        # FLAT
        # ====================================================

        if position == 0:

            buy = (
                high[k] >= hh[k]
            )

            sell = (
                low[k] <= ll[k]
            )

            # Both channel boundaries touched in same bar.
            if buy and sell:

                delta = (
                    -slpg
                    +
                    pv
                    *
                    (ll[k] - hh[k])
                )

            # Long breakout.
            elif buy:

                delta = (
                    -slpg / 2.0
                    +
                    pv
                    *
                    (close[k] - hh[k])
                )

                position = 1
                traded = True

                benchmark_long = high[k]

            # Short breakout.
            elif sell:

                delta = (
                    -slpg / 2.0
                    -
                    pv
                    *
                    (close[k] - ll[k])
                )

                position = -1
                traded = True

                benchmark_short = low[k]

        # ====================================================
        # LONG
        # ====================================================

        if (
            position == 1
            and
            not traded
        ):

            sell_short = (
                low[k] <= ll[k]
            )

            sell = (
                low[k]
                <=
                benchmark_long
                *
                (1.0 - stop_pct)
            )

            # ------------------------------------------------
            # Both trailing stop and opposite channel touched.
            # ------------------------------------------------

            if (
                sell_short
                and
                sell
            ):

                delta = (
                    delta
                    -
                    slpg
                    -
                    2.0
                    *
                    pv
                    *
                    (close[k] - ll[k])
                )

                position = -1

                benchmark_short = low[k]

            else:

                # --------------------------------------------
                # Trailing-stop exit.
                # --------------------------------------------

                if sell:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        -
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_long
                            *
                            (1.0 - stop_pct)
                        )
                    )

                    position = 0

                # --------------------------------------------
                # Opposite-channel reversal.
                # --------------------------------------------

                if sell_short:

                    delta = (
                        delta
                        -
                        slpg
                        -
                        2.0
                        *
                        pv
                        *
                        (close[k] - ll[k])
                    )

                    position = -1

                    benchmark_short = low[k]

            # Update long trailing benchmark.
            if high[k] > benchmark_long:
                benchmark_long = high[k]

        # ====================================================
        # SHORT
        # ====================================================

        if (
            position == -1
            and
            not traded
        ):

            buy_long = (
                high[k] >= hh[k]
            )

            buy = (
                high[k]
                >=
                benchmark_short
                *
                (1.0 + stop_pct)
            )

            # ------------------------------------------------
            # Both trailing stop and opposite channel touched.
            # ------------------------------------------------

            if (
                buy_long
                and
                buy
            ):

                delta = (
                    delta
                    -
                    slpg
                    +
                    2.0
                    *
                    pv
                    *
                    (close[k] - hh[k])
                )

                position = 1

                benchmark_long = high[k]

            else:

                # --------------------------------------------
                # Trailing-stop exit.
                # --------------------------------------------

                if buy:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        +
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_short
                            *
                            (1.0 + stop_pct)
                        )
                    )

                    position = 0

                # --------------------------------------------
                # Opposite-channel reversal.
                # --------------------------------------------

                if buy_long:

                    delta = (
                        delta
                        -
                        slpg
                        +
                        2.0
                        *
                        pv
                        *
                        (close[k] - hh[k])
                    )

                    position = 1

                    benchmark_long = high[k]

            # Update short trailing benchmark.
            if low[k] < benchmark_short:
                benchmark_short = low[k]

        # ====================================================
        # EQUITY / DRAWDOWN
        # ====================================================

        equity += delta

        if equity > equity_max:
            equity_max = equity

        dd = (
            equity
            -
            equity_max
        )

        if dd < min_dd:
            min_dd = dd

    net_profit = (
        equity
        -
        initial_equity
    )

    if min_dd < 0.0:

        objective = (
            net_profit
            /
            abs(min_dd)
        )

    elif net_profit > 0.0:

        objective = net_profit

    else:

        objective = -np.inf

    return (
        net_profit,
        min_dd,
        objective
    )


# ============================================================
# 2.3 Parallel StpPct evaluation
# ============================================================

@njit(parallel=True)
def score_stop_grid_parallel(
    high,
    low,
    close,
    hh,
    ll,
    length,
    stop_grid,
    slpg,
    pv,
    initial_equity
):

    m = len(stop_grid)

    profits = np.empty(
        m,
        dtype=np.float64
    )

    drawdowns = np.empty(
        m,
        dtype=np.float64
    )

    objectives = np.empty(
        m,
        dtype=np.float64
    )

    for j in prange(m):

        (
            profit,
            drawdown,
            objective
        ) = score_strategy_fast(
            high,
            low,
            close,
            hh,
            ll,
            length,
            stop_grid[j],
            slpg,
            pv,
            initial_equity
        )

        profits[j] = profit
        drawdowns[j] = drawdown
        objectives[j] = objective

    return (
        profits,
        drawdowns,
        objectives
    )


# ============================================================
# 2.4 Exact full-grid optimizer
# ============================================================

def optimize_full_grid_final(
    high,
    low,
    close,
    chn_grid,
    stop_grid,
    pv,
    slpg,
    initial_equity,
    show_progress=False
):

    best_objective = -np.inf

    best_length = None
    best_stop = None
    best_profit = None
    best_drawdown = None

    start_time = time.perf_counter()

    iterator = chn_grid

    if show_progress:
        iterator = tqdm(
            chn_grid,
            desc="Full-grid optimization"
        )

    for length in iterator:

        length = int(length)

        # ----------------------------------------------------
        # Construct prior-bar channels once per ChnLen.
        # ----------------------------------------------------

        hh, ll = build_channels_deque(
            high,
            low,
            length
        )

        # ----------------------------------------------------
        # Evaluate all StpPct values.
        # ----------------------------------------------------

        (
            profits,
            drawdowns,
            objectives
        ) = score_stop_grid_parallel(
            high,
            low,
            close,
            hh,
            ll,
            length,
            stop_grid,
            slpg,
            pv,
            initial_equity
        )

        # ----------------------------------------------------
        # Best stop for this ChnLen.
        # ----------------------------------------------------

        local_idx = int(
            np.argmax(objectives)
        )

        local_objective = float(
            objectives[local_idx]
        )

        # ----------------------------------------------------
        # Update global optimum.
        # ----------------------------------------------------

        if local_objective > best_objective:

            best_objective = (
                local_objective
            )

            best_length = (
                length
            )

            best_stop = float(
                stop_grid[local_idx]
            )

            best_profit = float(
                profits[local_idx]
            )

            best_drawdown = float(
                drawdowns[local_idx]
            )

    elapsed = (
        time.perf_counter()
        -
        start_time
    )

    return {
        "ChnLen": best_length,
        "StpPct": best_stop,
        "NetProfit": best_profit,
        "MaxDrawdown": best_drawdown,
        "Objective": best_objective,
        "RuntimeSeconds": elapsed,
        "Evaluations": (
            len(chn_grid)
            *
            len(stop_grid)
        ),
    }


# ============================================================
# 2.5 Strategy-path engine
#
# Needed later for IS-conditioned OOS evaluation.
# Must use exactly the same accounting as the scorer.
# ============================================================

@njit
def run_strategy_path_fast(
    high,
    low,
    close,
    length,
    stop_pct,
    slpg,
    pv
):

    n = len(close)

    hh, ll = build_channels_deque(
        high,
        low,
        length
    )

    pnl = np.zeros(
        n,
        dtype=np.float64
    )

    position_path = np.zeros(
        n,
        dtype=np.int8
    )

    position = 0

    benchmark_long = 0.0
    benchmark_short = 0.0

    for k in range(
        length,
        n
    ):

        traded = False

        delta = (
            pv
            *
            (close[k] - close[k - 1])
            *
            position
        )

        # ====================================================
        # FLAT
        # ====================================================

        if position == 0:

            buy = (
                high[k] >= hh[k]
            )

            sell = (
                low[k] <= ll[k]
            )

            if buy and sell:

                delta = (
                    -slpg
                    +
                    pv
                    *
                    (ll[k] - hh[k])
                )

            elif buy:

                delta = (
                    -slpg / 2.0
                    +
                    pv
                    *
                    (close[k] - hh[k])
                )

                position = 1
                traded = True

                benchmark_long = high[k]

            elif sell:

                delta = (
                    -slpg / 2.0
                    -
                    pv
                    *
                    (close[k] - ll[k])
                )

                position = -1
                traded = True

                benchmark_short = low[k]

        # ====================================================
        # LONG
        # ====================================================

        if (
            position == 1
            and
            not traded
        ):

            sell_short = (
                low[k] <= ll[k]
            )

            sell = (
                low[k]
                <=
                benchmark_long
                *
                (1.0 - stop_pct)
            )

            if (
                sell_short
                and
                sell
            ):

                delta = (
                    delta
                    -
                    slpg
                    -
                    2.0
                    *
                    pv
                    *
                    (close[k] - ll[k])
                )

                position = -1

                benchmark_short = low[k]

            else:

                if sell:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        -
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_long
                            *
                            (1.0 - stop_pct)
                        )
                    )

                    position = 0

                if sell_short:

                    delta = (
                        delta
                        -
                        slpg
                        -
                        2.0
                        *
                        pv
                        *
                        (close[k] - ll[k])
                    )

                    position = -1

                    benchmark_short = low[k]

            if high[k] > benchmark_long:
                benchmark_long = high[k]

        # ====================================================
        # SHORT
        # ====================================================

        if (
            position == -1
            and
            not traded
        ):

            buy_long = (
                high[k] >= hh[k]
            )

            buy = (
                high[k]
                >=
                benchmark_short
                *
                (1.0 + stop_pct)
            )

            if (
                buy_long
                and
                buy
            ):

                delta = (
                    delta
                    -
                    slpg
                    +
                    2.0
                    *
                    pv
                    *
                    (close[k] - hh[k])
                )

                position = 1

                benchmark_long = high[k]

            else:

                if buy:

                    delta = (
                        delta
                        -
                        slpg / 2.0
                        +
                        pv
                        *
                        (
                            close[k]
                            -
                            benchmark_short
                            *
                            (1.0 + stop_pct)
                        )
                    )

                    position = 0

                if buy_long:

                    delta = (
                        delta
                        -
                        slpg
                        +
                        2.0
                        *
                        pv
                        *
                        (close[k] - hh[k])
                    )

                    position = 1

                    benchmark_long = high[k]

            if low[k] < benchmark_short:
                benchmark_short = low[k]

        # ====================================================
        # STORE BAR-LEVEL RESULT
        # ====================================================

        pnl[k] = delta

        position_path[k] = position

    return (
        pnl,
        position_path
    )


# ============================================================
# Configuration audit
# ============================================================

print("=" * 72)
print("TASK M — EXACT FINALIZED TASK K ENGINE")
print("=" * 72)

print(f"Bars loaded            : {N:,}")
print(f"ChnLen values          : {len(CHNLEN_GRID):,}")
print(f"StpPct values          : {len(STPPCT_GRID):,}")
print(f"Full-grid evaluations  : {N_GRID_COMBINATIONS:,}")
print(f"Point value            : {PV:,.2f}")
print(f"Round-turn slippage    : {SLPG:.3f}")
print(f"Initial equity         : {E0:,.2f}")

print()
print("Exact Task K scorer, optimizer, and path engine initialized.")

TASK M — EXACT FINALIZED TASK K ENGINE
Bars loaded            : 138,744
ChnLen values          : 951
StpPct values          : 96
Full-grid evaluations  : 91,296
Point value            : 1,000.00
Round-turn slippage    : 0.065
Initial equity         : 100,000.00

Exact Task K scorer, optimizer, and path engine initialized.


## 3. Baseline Engine Reconciliation with Task K

Before running the sensitivity analysis, the Task M implementation is reconciled against the finalized Task K baseline.

The first `(T = 4 years, τ = 3 months)` walk-forward window is reconstructed using the same AUG data, full parameter grid, strategy engine, transaction-cost assumptions, and optimization objective.

This is a replication check rather than a new optimization experiment. Task M should reproduce the finalized Task K first-window optimum before any alternative \(T/\tau\) specifications are evaluated.

In [4]:
BASELINE_T_YEARS = 4
BASELINE_TAU_MONTHS = 3

# Task K's first OOS quarter begins on 2022-07-01.
FIRST_OOS_START = pd.Timestamp("2022-07-01 00:00:00")

FIRST_IS_START = FIRST_OOS_START - pd.DateOffset(
    years=BASELINE_T_YEARS
)

FIRST_IS_END = FIRST_OOS_START


# ------------------------------------------------------------
# Extract the exact first IS sample
#
# IS interval:
# [FIRST_IS_START, FIRST_OOS_START)
# ------------------------------------------------------------

first_is = aug.loc[
    (aug.index >= FIRST_IS_START)
    &
    (aug.index < FIRST_IS_END)
].copy()

if len(first_is) == 0:
    raise RuntimeError(
        "The reconstructed first Task K IS window is empty."
    )


first_high = first_is["High"].to_numpy(dtype=np.float64)
first_low = first_is["Low"].to_numpy(dtype=np.float64)
first_close = first_is["Close"].to_numpy(dtype=np.float64)


# ------------------------------------------------------------
# Run the exact full-grid optimization
# ------------------------------------------------------------

print("=" * 72)
print("TASK K BASELINE RECONCILIATION — FIRST IS WINDOW")
print("=" * 72)

print(f"IS requested start     : {FIRST_IS_START}")
print(f"IS requested end       : {FIRST_IS_END} (exclusive)")
print(f"IS actual first bar    : {first_is.index.min()}")
print(f"IS actual last bar     : {first_is.index.max()}")
print(f"IS bars                : {len(first_is):,}")

print()
print(f"Grid combinations      : {N_GRID_COMBINATIONS:,}")
print("Running exact full-grid optimization...")

t0 = time.perf_counter()

first_result = optimize_full_grid_final(
    high=first_high,
    low=first_low,
    close=first_close,
    chn_grid=CHNLEN_GRID,
    stop_grid=STPPCT_GRID,
    pv=PV,
    slpg=SLPG,
    initial_equity=E0,
)

elapsed = time.perf_counter() - t0


# ------------------------------------------------------------
# Finalized Task K first-window reference
# ------------------------------------------------------------

TASK_K_REFERENCE = {
    "ChnLen": 500,
    "StpPct": 0.005,
    "NetProfit": 421_702.10,
}


# ------------------------------------------------------------
# Reconciliation report
# ------------------------------------------------------------

print()
print("=" * 72)
print("OPTIMIZATION RESULT")
print("=" * 72)

print(f"Optimal ChnLen         : {first_result['ChnLen']}")
print(f"Optimal StpPct         : {first_result['StpPct']:.3f}")
print(f"Net Profit             : {first_result['NetProfit']:,.6f}")
print(f"Maximum Drawdown       : {first_result['MaxDrawdown']:,.6f}")
print(f"Objective              : {first_result['Objective']:.6f}")
print(f"Evaluations            : {first_result['Evaluations']:,}")
print(f"Runtime                : {elapsed:.2f} seconds")


# ------------------------------------------------------------
# Hard reconciliation checks
# ------------------------------------------------------------

chn_match = (
    first_result["ChnLen"]
    == TASK_K_REFERENCE["ChnLen"]
)

stop_match = np.isclose(
    first_result["StpPct"],
    TASK_K_REFERENCE["StpPct"],
    atol=1e-12,
)

profit_error = (
    first_result["NetProfit"]
    - TASK_K_REFERENCE["NetProfit"]
)

profit_match = np.isclose(
    first_result["NetProfit"],
    TASK_K_REFERENCE["NetProfit"],
    atol=1e-6,
)

evaluation_match = (
    first_result["Evaluations"]
    == N_GRID_COMBINATIONS
)


print()
print("=" * 72)
print("TASK K RECONCILIATION CHECKS")
print("=" * 72)

print(f"ChnLen match           : {chn_match}")
print(f"StpPct match           : {stop_match}")
print(f"Net Profit error       : {profit_error:.12f}")
print(f"Net Profit match       : {profit_match}")
print(f"Full-grid count match  : {evaluation_match}")


assert chn_match, (
    "Task M does not reproduce the finalized Task K ChnLen."
)

assert stop_match, (
    "Task M does not reproduce the finalized Task K StpPct."
)

assert profit_match, (
    "Task M does not reproduce the finalized Task K Net Profit."
)

assert evaluation_match, (
    "Task M did not evaluate the complete professor-specified grid."
)


print()
print("TASK M ENGINE SUCCESSFULLY RECONCILED WITH TASK K BASELINE.")

TASK K BASELINE RECONCILIATION — FIRST IS WINDOW
IS requested start     : 2018-07-01 00:00:00
IS requested end       : 2022-07-01 00:00:00 (exclusive)
IS actual first bar    : 2018-07-02 09:05:00
IS actual last bar     : 2022-06-30 15:00:00
IS bars                : 69,912

Grid combinations      : 91,296
Running exact full-grid optimization...

OPTIMIZATION RESULT
Optimal ChnLen         : 500
Optimal StpPct         : 0.005
Net Profit             : 421,702.092500
Maximum Drawdown       : -8,792.180000
Objective              : 47.963314
Evaluations            : 91,296
Runtime                : 7.24 seconds

TASK K RECONCILIATION CHECKS
ChnLen match           : True
StpPct match           : True
Net Profit error       : -0.007499995816
Net Profit match       : True
Full-grid count match  : True

TASK M ENGINE SUCCESSFULLY RECONCILED WITH TASK K BASELINE.


In [5]:
BASELINE_T_YEARS = 4
BASELINE_TAU_MONTHS = 3

# Task K's first OOS quarter begins on 2022-07-01.
FIRST_OOS_START = pd.Timestamp("2022-07-01 00:00:00")

FIRST_IS_START = (
    FIRST_OOS_START
    -
    pd.DateOffset(years=BASELINE_T_YEARS)
)

FIRST_IS_END = FIRST_OOS_START


# ------------------------------------------------------------
# Extract the exact first IS sample
#
# IS interval:
# [FIRST_IS_START, FIRST_OOS_START)
# ------------------------------------------------------------

first_is = aug.loc[
    (aug.index >= FIRST_IS_START)
    &
    (aug.index < FIRST_IS_END)
].copy()

if len(first_is) == 0:
    raise RuntimeError(
        "The reconstructed first Task K IS window is empty."
    )


first_high = np.ascontiguousarray(
    first_is["High"].to_numpy(dtype=np.float64)
)

first_low = np.ascontiguousarray(
    first_is["Low"].to_numpy(dtype=np.float64)
)

first_close = np.ascontiguousarray(
    first_is["Close"].to_numpy(dtype=np.float64)
)


# ------------------------------------------------------------
# Run the exact full-grid optimization
# ------------------------------------------------------------

print("=" * 72)
print("TASK K BASELINE RECONCILIATION — FIRST IS WINDOW")
print("=" * 72)

print(f"IS requested start     : {FIRST_IS_START}")
print(f"IS requested end       : {FIRST_IS_END} (exclusive)")
print(f"IS actual first bar    : {first_is.index.min()}")
print(f"IS actual last bar     : {first_is.index.max()}")
print(f"IS bars                : {len(first_is):,}")

print()
print(f"Grid combinations      : {N_GRID_COMBINATIONS:,}")
print("Running exact full-grid optimization...")

t0 = time.perf_counter()

first_result = optimize_full_grid_final(
    high=first_high,
    low=first_low,
    close=first_close,
    chn_grid=CHNLEN_GRID,
    stop_grid=STPPCT_GRID,
    pv=PV,
    slpg=SLPG,
    initial_equity=E0,
    show_progress=False,
)

elapsed = (
    time.perf_counter()
    -
    t0
)


# ------------------------------------------------------------
# Exact finalized Task K first-window reference
# ------------------------------------------------------------

TASK_K_REFERENCE = {
    "ChnLen": 500,
    "StpPct": 0.005,
    "NetProfit": 421_702.0925,
    "MaxDrawdown": -8_792.18,
}


# ------------------------------------------------------------
# Reconciliation report
# ------------------------------------------------------------

print()
print("=" * 72)
print("OPTIMIZATION RESULT")
print("=" * 72)

print(f"Optimal ChnLen         : {first_result['ChnLen']}")
print(f"Optimal StpPct         : {first_result['StpPct']:.3f}")
print(f"Net Profit             : {first_result['NetProfit']:,.6f}")
print(f"Maximum Drawdown       : {first_result['MaxDrawdown']:,.6f}")
print(f"Objective              : {first_result['Objective']:.6f}")
print(f"Evaluations            : {first_result['Evaluations']:,}")
print(f"Runtime                : {elapsed:.2f} seconds")


# ------------------------------------------------------------
# Exact reconciliation checks
# ------------------------------------------------------------

chn_match = (
    first_result["ChnLen"]
    ==
    TASK_K_REFERENCE["ChnLen"]
)

stop_match = np.isclose(
    first_result["StpPct"],
    TASK_K_REFERENCE["StpPct"],
    rtol=0.0,
    atol=1e-12,
)

profit_error = (
    first_result["NetProfit"]
    -
    TASK_K_REFERENCE["NetProfit"]
)

profit_match = np.isclose(
    first_result["NetProfit"],
    TASK_K_REFERENCE["NetProfit"],
    rtol=0.0,
    atol=1e-8,
)

drawdown_error = (
    first_result["MaxDrawdown"]
    -
    TASK_K_REFERENCE["MaxDrawdown"]
)

drawdown_match = np.isclose(
    first_result["MaxDrawdown"],
    TASK_K_REFERENCE["MaxDrawdown"],
    rtol=0.0,
    atol=1e-8,
)

evaluation_match = (
    first_result["Evaluations"]
    ==
    N_GRID_COMBINATIONS
)


print()
print("=" * 72)
print("TASK K RECONCILIATION CHECKS")
print("=" * 72)

print(f"ChnLen match           : {chn_match}")
print(f"StpPct match           : {stop_match}")
print(f"Net Profit error       : {profit_error:.12f}")
print(f"Net Profit match       : {profit_match}")
print(f"Drawdown error         : {drawdown_error:.12f}")
print(f"Drawdown match         : {drawdown_match}")
print(f"Full-grid count match  : {evaluation_match}")


assert chn_match
assert stop_match
assert profit_match
assert drawdown_match
assert evaluation_match


print()
print(
    "TASK M ENGINE SUCCESSFULLY RECONCILED "
    "WITH TASK K BASELINE."
)

TASK K BASELINE RECONCILIATION — FIRST IS WINDOW
IS requested start     : 2018-07-01 00:00:00
IS requested end       : 2022-07-01 00:00:00 (exclusive)
IS actual first bar    : 2018-07-02 09:05:00
IS actual last bar     : 2022-06-30 15:00:00
IS bars                : 69,912

Grid combinations      : 91,296
Running exact full-grid optimization...

OPTIMIZATION RESULT
Optimal ChnLen         : 500
Optimal StpPct         : 0.005
Net Profit             : 421,702.092500
Maximum Drawdown       : -8,792.180000
Objective              : 47.963314
Evaluations            : 91,296
Runtime                : 7.15 seconds

TASK K RECONCILIATION CHECKS
ChnLen match           : True
StpPct match           : True
Net Profit error       : 0.000000004133
Net Profit match       : True
Drawdown error         : 0.000000000065
Drawdown match         : True
Full-grid count match  : True

TASK M ENGINE SUCCESSFULLY RECONCILED WITH TASK K BASELINE.


## 4. Walk-Forward Calendars for the Sensitivity Specifications

Each sensitivity specification uses the same calendar-based walk-forward construction as the finalized Task K baseline.

For a given in-sample length \(T\) and OOS horizon \(\tau\):

1. the first eligible OOS period begins at the first standard calendar-quarter boundary for which a complete \(T\)-year in-sample history is available;
2. the immediately preceding \(T\) years are used for parameter optimization;
3. the selected parameters are evaluated over the subsequent \(\tau\)-month OOS interval;
4. OOS intervals are non-overlapping;
5. only complete OOS windows contained within the available AUG history are retained.

At this stage, the six calendars are constructed and audited before any sensitivity optimization is performed.

In [6]:
DATA_START = aug.index.min()
DATA_END = aug.index.max()


def build_walk_forward_calendar(
    data_index,
    T_years,
    tau_months
):
    """
    Construct non-overlapping calendar-based walk-forward windows.

    Parameters
    ----------
    data_index : pd.DatetimeIndex
        Full AUG timestamp index.

    T_years : int
        Length of the in-sample optimization window in years.

    tau_months : int
        Length of each non-overlapping OOS evaluation window
        in months.

    Returns
    -------
    pd.DataFrame
        One row per valid walk-forward window.
    """

    data_start = data_index.min()
    data_end = data_index.max()

    # --------------------------------------------------------
    # Earliest date at which a complete T-year IS history
    # can exist.
    # --------------------------------------------------------

    earliest_oos_date = (
        data_start
        +
        pd.DateOffset(years=T_years)
    )

    # --------------------------------------------------------
    # Candidate standard calendar-quarter boundaries.
    # --------------------------------------------------------

    candidate_boundaries = pd.date_range(
        start=pd.Timestamp(
            year=earliest_oos_date.year,
            month=1,
            day=1
        ),
        end=data_end.normalize(),
        freq="QS"
    )

    candidate_boundaries = candidate_boundaries[
        candidate_boundaries
        >=
        earliest_oos_date.normalize()
    ]

    # --------------------------------------------------------
    # For tau = 6 months, retain non-overlapping semiannual
    # windows by stepping through every second quarter.
    #
    # More generally, tau must be a multiple of 3 months.
    # --------------------------------------------------------

    if tau_months % 3 != 0:
        raise ValueError(
            "tau_months must be a multiple of 3."
        )

    boundary_step = (
        tau_months // 3
    )

    candidate_boundaries = candidate_boundaries[
        ::boundary_step
    ]

    # --------------------------------------------------------
    # Construct valid complete windows.
    # --------------------------------------------------------

    rows = []

    for window_id, oos_start in enumerate(
        candidate_boundaries,
        start=1
    ):

        oos_end = (
            oos_start
            +
            pd.DateOffset(months=tau_months)
        )

        # Require the complete OOS calendar interval to lie
        # inside the available AUG history.
        if oos_end > data_end:
            continue

        is_start = (
            oos_start
            -
            pd.DateOffset(years=T_years)
        )

        is_mask = (
            (data_index >= is_start)
            &
            (data_index < oos_start)
        )

        oos_mask = (
            (data_index >= oos_start)
            &
            (data_index < oos_end)
        )

        is_positions = np.flatnonzero(
            np.asarray(is_mask)
        )

        oos_positions = np.flatnonzero(
            np.asarray(oos_mask)
        )

        if (
            len(is_positions) == 0
            or
            len(oos_positions) == 0
        ):
            continue

        # Require actual observations at or after the intended
        # IS start, and a complete calendar T-year history.
        actual_is_start = data_index[
            is_positions[0]
        ]

        actual_is_end = data_index[
            is_positions[-1]
        ]

        actual_oos_start = data_index[
            oos_positions[0]
        ]

        actual_oos_end = data_index[
            oos_positions[-1]
        ]

        rows.append(
            {
                "Window": window_id,
                "T_years": T_years,
                "tau_months": tau_months,

                "IS_Start": is_start,
                "IS_End": oos_start,

                "IS_Actual_Start": actual_is_start,
                "IS_Actual_End": actual_is_end,
                "IS_Bars": len(is_positions),

                "OOS_Start": oos_start,
                "OOS_End": oos_end,

                "OOS_Actual_Start": actual_oos_start,
                "OOS_Actual_End": actual_oos_end,
                "OOS_Bars": len(oos_positions),
            }
        )

    calendar = pd.DataFrame(rows)

    if len(calendar) == 0:
        raise RuntimeError(
            f"No valid windows for "
            f"T={T_years}, tau={tau_months}."
        )

    # Renumber after any incomplete final windows were removed.
    calendar["Window"] = np.arange(
        1,
        len(calendar) + 1
    )

    return calendar


# ------------------------------------------------------------
# Build all six calendars
# ------------------------------------------------------------

SENSITIVITY_CALENDARS = {}

calendar_summary_rows = []

for spec in SENSITIVITY_SPECS:

    label = spec["label"]

    calendar = build_walk_forward_calendar(
        data_index=aug.index,
        T_years=spec["T_years"],
        tau_months=spec["tau_months"],
    )

    SENSITIVITY_CALENDARS[label] = calendar

    calendar_summary_rows.append(
        {
            "Specification": label,
            "T_years": spec["T_years"],
            "tau_months": spec["tau_months"],
            "Baseline": spec["baseline"],
            "Windows": len(calendar),

            "First_OOS_Start":
                calendar["OOS_Actual_Start"].iloc[0],

            "Last_OOS_End":
                calendar["OOS_Actual_End"].iloc[-1],

            "Median_IS_Bars":
                int(calendar["IS_Bars"].median()),

            "Median_OOS_Bars":
                int(calendar["OOS_Bars"].median()),
        }
    )


calendar_summary = pd.DataFrame(
    calendar_summary_rows
)


# ------------------------------------------------------------
# Calendar audit
# ------------------------------------------------------------

print("=" * 88)
print("TASK M — WALK-FORWARD CALENDAR SUMMARY")
print("=" * 88)

display(calendar_summary)


# ------------------------------------------------------------
# Baseline Task K reconciliation
# ------------------------------------------------------------

baseline_calendar = (
    SENSITIVITY_CALENDARS["T4_tau3"]
)

baseline_window_count_match = (
    len(baseline_calendar)
    ==
    15
)

baseline_first_oos_match = (
    baseline_calendar["OOS_Actual_Start"].iloc[0]
    ==
    pd.Timestamp("2022-07-01 09:05:00")
)

baseline_last_oos_match = (
    baseline_calendar["OOS_Actual_End"].iloc[-1]
    ==
    pd.Timestamp("2026-03-31 15:00:00")
)

baseline_first_is_bars_match = (
    int(
        baseline_calendar["IS_Bars"].iloc[0]
    )
    ==
    69_912
)


print()
print("=" * 88)
print("TASK K BASELINE CALENDAR RECONCILIATION")
print("=" * 88)

print(
    f"Baseline windows       : "
    f"{len(baseline_calendar)}"
)

print(
    f"First OOS actual start : "
    f"{baseline_calendar['OOS_Actual_Start'].iloc[0]}"
)

print(
    f"Last OOS actual end    : "
    f"{baseline_calendar['OOS_Actual_End'].iloc[-1]}"
)

print(
    f"First IS bars          : "
    f"{int(baseline_calendar['IS_Bars'].iloc[0]):,}"
)

print()
print(
    f"Window count match     : "
    f"{baseline_window_count_match}"
)

print(
    f"First OOS start match  : "
    f"{baseline_first_oos_match}"
)

print(
    f"Last OOS end match     : "
    f"{baseline_last_oos_match}"
)

print(
    f"First IS bars match    : "
    f"{baseline_first_is_bars_match}"
)


assert baseline_window_count_match
assert baseline_first_oos_match
assert baseline_last_oos_match
assert baseline_first_is_bars_match


# ------------------------------------------------------------
# Non-overlap validation for every specification
# ------------------------------------------------------------

non_overlap_checks = {}

for label, calendar in SENSITIVITY_CALENDARS.items():

    if len(calendar) <= 1:
        non_overlap = True

    else:
        previous_end = (
            calendar["OOS_End"]
            .iloc[:-1]
            .reset_index(drop=True)
        )

        next_start = (
            calendar["OOS_Start"]
            .iloc[1:]
            .reset_index(drop=True)
        )

        non_overlap = bool(
            (next_start >= previous_end).all()
        )

    non_overlap_checks[label] = non_overlap


print()
print("=" * 88)
print("NON-OVERLAPPING OOS WINDOW CHECKS")
print("=" * 88)

for label, passed in non_overlap_checks.items():
    print(
        f"{label:10s}: {passed}"
    )

assert all(
    non_overlap_checks.values()
)


print()
print(
    "ALL TASK M WALK-FORWARD CALENDARS "
    "SUCCESSFULLY CONSTRUCTED AND VALIDATED."
)

TASK M — WALK-FORWARD CALENDAR SUMMARY


,Specification,T_years,tau_months,Baseline,Windows,First_OOS_Start,Last_OOS_End,Median_IS_Bars,Median_OOS_Bars
0,T4_tau3,4,3,True,15,2022-07-01 09:05:00,2026-03-31 15:00:00,69840,4320
1,T4_tau6,4,6,False,7,2022-07-01 09:05:00,2025-12-31 15:00:00,69840,8928
2,T5_tau3,5,3,False,11,2023-07-03 09:05:00,2026-03-31 15:00:00,87264,4320
3,T5_tau6,5,6,False,5,2023-07-03 09:05:00,2025-12-31 15:00:00,87336,8928
4,T6_tau3,6,3,False,7,2024-07-01 09:05:00,2026-03-31 15:00:00,104760,4320
5,T6_tau6,6,6,False,3,2024-07-01 09:05:00,2025-12-31 15:00:00,104760,9000



TASK K BASELINE CALENDAR RECONCILIATION
Baseline windows       : 15
First OOS actual start : 2022-07-01 09:05:00
Last OOS actual end    : 2026-03-31 15:00:00
First IS bars          : 69,912

Window count match     : True
First OOS start match  : True
Last OOS end match     : True
First IS bars match    : True

NON-OVERLAPPING OOS WINDOW CHECKS
T4_tau3   : True
T4_tau6   : True
T5_tau3   : True
T5_tau6   : True
T6_tau3   : True
T6_tau6   : True

ALL TASK M WALK-FORWARD CALENDARS SUCCESSFULLY CONSTRUCTED AND VALIDATED.


## 5. Run the Full Walk-Forward Sensitivity Analysis

Each of the six \(T/\tau\) specifications is now evaluated using the same exact procedure as the finalized Task K baseline.

For every walk-forward window:

1. the corresponding in-sample interval is extracted;
2. all 91,296 parameter combinations are evaluated;
3. the pair maximizing `Net Profit / |Maximum Drawdown|` is selected using in-sample data only;
4. the strategy is then run from the beginning of that in-sample interval through the end of the corresponding OOS interval;
5. only bar-level P&L generated inside the OOS interval is retained;
6. non-overlapping OOS segments are stitched chronologically.

This preserves the IS-conditioned OOS methodology used in Task K. No parameter information from the OOS interval is used in optimization.

The purpose remains sensitivity analysis rather than ex-post selection of a preferred \(T/\tau\) specification.

In [7]:
SENSITIVITY_OPTIMIZATIONS = {}
SENSITIVITY_OOS_PATHS = {}
SENSITIVITY_WINDOW_RESULTS = {}

spec_summary_rows = []


# ------------------------------------------------------------
# Helper: convert calendar boundary to array position
# ------------------------------------------------------------

index_values = aug.index.to_numpy(
    dtype="datetime64[ns]"
)


def timestamp_to_position(ts):
    """
    Return the first array position whose timestamp is
    greater than or equal to ts.
    """

    return int(
        np.searchsorted(
            index_values,
            np.datetime64(pd.Timestamp(ts)),
            side="left"
        )
    )


# ------------------------------------------------------------
# Main sensitivity loop
# ------------------------------------------------------------

grand_start_time = time.perf_counter()


for spec_number, spec in enumerate(
    SENSITIVITY_SPECS,
    start=1
):

    label = spec["label"]

    calendar = (
        SENSITIVITY_CALENDARS[label]
        .copy()
        .reset_index(drop=True)
    )

    print()
    print("=" * 88)
    print(
        f"SPECIFICATION {spec_number}/{len(SENSITIVITY_SPECS)} "
        f"— {label}"
    )
    print("=" * 88)

    print(
        f"T = {spec['T_years']} years, "
        f"tau = {spec['tau_months']} months"
    )

    print(
        f"OOS windows: {len(calendar)}"
    )

    if spec["baseline"]:
        print("Status     : Task K baseline")

    else:
        print("Status     : Sensitivity specification")

    print()


    # ========================================================
    # Containers for this specification
    # ========================================================

    optimization_records = []
    window_records = []
    oos_segments = []

    spec_start_time = time.perf_counter()


    # ========================================================
    # Loop through walk-forward windows
    # ========================================================

    for i, row in calendar.iterrows():

        # ----------------------------------------------------
        # Convert calendar boundaries to exact array indices
        # ----------------------------------------------------

        is_start_idx = timestamp_to_position(
            row["IS_Start"]
        )

        oos_start_idx = timestamp_to_position(
            row["OOS_Start"]
        )

        oos_end_idx = timestamp_to_position(
            row["OOS_End"]
        )


        # ----------------------------------------------------
        # Exact IS sample
        # ----------------------------------------------------

        high_is = np.ascontiguousarray(
            HIGH[
                is_start_idx:oos_start_idx
            ],
            dtype=np.float64
        )

        low_is = np.ascontiguousarray(
            LOW[
                is_start_idx:oos_start_idx
            ],
            dtype=np.float64
        )

        close_is = np.ascontiguousarray(
            CLOSE[
                is_start_idx:oos_start_idx
            ],
            dtype=np.float64
        )


        # ----------------------------------------------------
        # Calendar / slicing reconciliation
        # ----------------------------------------------------

        assert (
            len(close_is)
            ==
            int(row["IS_Bars"])
        )


        # ----------------------------------------------------
        # Full 91,296-combination IS optimization
        # ----------------------------------------------------

        optimization_start = (
            time.perf_counter()
        )

        optimum = optimize_full_grid_final(
            high=high_is,
            low=low_is,
            close=close_is,
            chn_grid=CHNLEN_GRID,
            stop_grid=STPPCT_GRID,
            pv=PV,
            slpg=SLPG,
            initial_equity=E0,
            show_progress=False,
        )

        optimization_runtime = (
            time.perf_counter()
            -
            optimization_start
        )


        # ----------------------------------------------------
        # Selected IS parameters
        # ----------------------------------------------------

        length = int(
            optimum["ChnLen"]
        )

        stop_pct = float(
            optimum["StpPct"]
        )


        # ----------------------------------------------------
        # IS-conditioned path
        #
        # Run from IS start THROUGH OOS end.
        # This carries only information available before
        # and during that window.
        # ----------------------------------------------------

        high_path = np.ascontiguousarray(
            HIGH[
                is_start_idx:oos_end_idx
            ],
            dtype=np.float64
        )

        low_path = np.ascontiguousarray(
            LOW[
                is_start_idx:oos_end_idx
            ],
            dtype=np.float64
        )

        close_path = np.ascontiguousarray(
            CLOSE[
                is_start_idx:oos_end_idx
            ],
            dtype=np.float64
        )

        (
            path_pnl,
            position_path
        ) = run_strategy_path_fast(
            high=high_path,
            low=low_path,
            close=close_path,
            length=length,
            stop_pct=stop_pct,
            slpg=SLPG,
            pv=PV,
        )


        # ----------------------------------------------------
        # Extract ONLY OOS bar-level P&L
        # ----------------------------------------------------

        oos_local_start = (
            oos_start_idx
            -
            is_start_idx
        )

        oos_pnl = np.asarray(
            path_pnl[
                oos_local_start:
            ],
            dtype=np.float64
        )

        oos_position = np.asarray(
            position_path[
                oos_local_start:
            ],
            dtype=np.int8
        )

        oos_timestamps = (
            aug.index[
                oos_start_idx:oos_end_idx
            ]
        )


        # ----------------------------------------------------
        # Hard alignment checks
        # ----------------------------------------------------

        assert (
            len(oos_pnl)
            ==
            len(oos_timestamps)
        )

        assert (
            len(oos_pnl)
            ==
            int(row["OOS_Bars"])
        )

        assert np.isfinite(
            oos_pnl
        ).all()


        # ----------------------------------------------------
        # Window-level OOS statistics
        # ----------------------------------------------------

        oos_net_profit = float(
            oos_pnl.sum()
        )

        oos_cumulative = np.cumsum(
            oos_pnl
        )

        local_equity = (
            E0
            +
            oos_cumulative
        )

        local_peak = np.maximum.accumulate(
            np.concatenate(
                (
                    np.array(
                        [E0],
                        dtype=np.float64
                    ),
                    local_equity
                )
            )
        )[1:]

        local_drawdown = (
            local_equity
            -
            local_peak
        )

        oos_max_drawdown = float(
            local_drawdown.min()
            if len(local_drawdown) > 0
            else 0.0
        )


        # ----------------------------------------------------
        # Entry / ending position diagnostics
        # ----------------------------------------------------

        if oos_local_start > 0:

            entry_position = int(
                position_path[
                    oos_local_start - 1
                ]
            )

        else:

            entry_position = 0

        end_position = int(
            oos_position[-1]
            if len(oos_position) > 0
            else entry_position
        )


        # ----------------------------------------------------
        # Store bar-level OOS segment
        # ----------------------------------------------------

        segment = pd.DataFrame(
            {
                "Timestamp":
                    oos_timestamps,

                "P&L":
                    oos_pnl,

                "Position":
                    oos_position,

                "Window":
                    int(i + 1),

                "Specification":
                    label,

                "T_years":
                    int(spec["T_years"]),

                "tau_months":
                    int(spec["tau_months"]),

                "ChnLen":
                    length,

                "StpPct":
                    stop_pct,
            }
        )

        oos_segments.append(
            segment
        )


        # ----------------------------------------------------
        # Store optimization result
        # ----------------------------------------------------

        optimization_records.append(
            {
                "Window":
                    int(i + 1),

                "IS_Start":
                    row["IS_Actual_Start"],

                "IS_End":
                    row["IS_Actual_End"],

                "OOS_Start":
                    row["OOS_Actual_Start"],

                "OOS_End":
                    row["OOS_Actual_End"],

                "IS_Bars":
                    int(row["IS_Bars"]),

                "OOS_Bars":
                    int(row["OOS_Bars"]),

                "ChnLen":
                    length,

                "StpPct":
                    stop_pct,

                "IS_NetProfit":
                    float(
                        optimum["NetProfit"]
                    ),

                "IS_MaxDrawdown":
                    float(
                        optimum["MaxDrawdown"]
                    ),

                "IS_Objective":
                    float(
                        optimum["Objective"]
                    ),

                "Evaluations":
                    int(
                        optimum["Evaluations"]
                    ),

                "RuntimeSeconds":
                    float(
                        optimization_runtime
                    ),
            }
        )


        # ----------------------------------------------------
        # Store window OOS result
        # ----------------------------------------------------

        window_records.append(
            {
                "Window":
                    int(i + 1),

                "OOS_Start":
                    row["OOS_Actual_Start"],

                "OOS_End":
                    row["OOS_Actual_End"],

                "OOS_Bars":
                    len(oos_pnl),

                "ChnLen":
                    length,

                "StpPct":
                    stop_pct,

                "Entry_Position":
                    entry_position,

                "End_Position":
                    end_position,

                "OOS_NetProfit":
                    oos_net_profit,

                "OOS_MaxDrawdown":
                    oos_max_drawdown,
            }
        )


        # ----------------------------------------------------
        # Progress output
        # ----------------------------------------------------

        print(
            f"[{i + 1:02d}/{len(calendar):02d}] "
            f"{label} | "
            f"L={length:5d} | "
            f"S={stop_pct:.3f} | "
            f"IS Obj={float(optimum['Objective']):10.4f} | "
            f"OOS P&L={oos_net_profit:12,.2f} | "
            f"{optimization_runtime:6.2f}s"
        )


    # ========================================================
    # Combine this specification's results
    # ========================================================

    optimization_df = pd.DataFrame(
        optimization_records
    )

    window_df = pd.DataFrame(
        window_records
    )

    oos_df = pd.concat(
        oos_segments,
        ignore_index=True
    )


    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    oos_df = (
        oos_df
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # No overlapping / duplicate OOS timestamps
    # --------------------------------------------------------

    duplicate_oos = int(
        oos_df["Timestamp"]
        .duplicated()
        .sum()
    )

    assert duplicate_oos == 0


    # --------------------------------------------------------
    # Stitched OOS equity
    # --------------------------------------------------------

    oos_df["Equity"] = (
        E0
        +
        oos_df["P&L"].cumsum()
    )

    running_peak = (
        oos_df["Equity"]
        .cummax()
    )

    running_peak = np.maximum(
        running_peak,
        E0
    )

    oos_df["Drawdown"] = (
        oos_df["Equity"]
        -
        running_peak
    )


    # --------------------------------------------------------
    # Specification-level accounting
    # --------------------------------------------------------

    stitched_net_profit = float(
        oos_df["P&L"].sum()
    )

    ending_equity = float(
        oos_df["Equity"].iloc[-1]
    )

    accounting_error = (
        ending_equity
        -
        E0
        -
        stitched_net_profit
    )

    assert np.isclose(
        accounting_error,
        0.0,
        rtol=0.0,
        atol=1e-6
    )


    # --------------------------------------------------------
    # Additional checks
    # --------------------------------------------------------

    all_params_in_grid = bool(
        optimization_df["ChnLen"]
        .isin(CHNLEN_GRID)
        .all()
    )

    stop_grid_checks = np.array(
        [
            np.any(
                np.isclose(
                    STPPCT_GRID,
                    x,
                    rtol=0.0,
                    atol=1e-12
                )
            )
            for x
            in optimization_df["StpPct"]
        ],
        dtype=bool
    )

    all_stops_in_grid = bool(
        stop_grid_checks.all()
    )

    full_grid_checks = bool(
        (
            optimization_df["Evaluations"]
            ==
            N_GRID_COMBINATIONS
        )
        .all()
    )

    chronological = bool(
        oos_df["Timestamp"]
        .is_monotonic_increasing
    )


    assert all_params_in_grid
    assert all_stops_in_grid
    assert full_grid_checks
    assert chronological


    # --------------------------------------------------------
    # Save in memory
    # --------------------------------------------------------

    SENSITIVITY_OPTIMIZATIONS[
        label
    ] = optimization_df

    SENSITIVITY_WINDOW_RESULTS[
        label
    ] = window_df

    SENSITIVITY_OOS_PATHS[
        label
    ] = oos_df


    # --------------------------------------------------------
    # Save CSV outputs
    # --------------------------------------------------------

    optimization_df.to_csv(
        OUT_DIR
        /
        f"{label}_optimization.csv",
        index=False
    )

    window_df.to_csv(
        OUT_DIR
        /
        f"{label}_window_results.csv",
        index=False
    )

    oos_df.to_csv(
        OUT_DIR
        /
        f"{label}_oos_path.csv",
        index=False
    )


    # --------------------------------------------------------
    # Specification summary
    # --------------------------------------------------------

    spec_runtime = (
        time.perf_counter()
        -
        spec_start_time
    )

    positive_windows = int(
        (
            window_df["OOS_NetProfit"]
            >
            0.0
        ).sum()
    )

    negative_windows = int(
        (
            window_df["OOS_NetProfit"]
            <
            0.0
        ).sum()
    )

    flat_windows = int(
        (
            window_df["OOS_NetProfit"]
            ==
            0.0
        ).sum()
    )

    lower_stop_share = float(
        np.mean(
            np.isclose(
                optimization_df["StpPct"],
                STPPCT_GRID[0],
                rtol=0.0,
                atol=1e-12
            )
        )
    )

    spec_summary_rows.append(
        {
            "Specification":
                label,

            "T_years":
                spec["T_years"],

            "tau_months":
                spec["tau_months"],

            "Baseline":
                spec["baseline"],

            "Windows":
                len(window_df),

            "OOS_Bars":
                len(oos_df),

            "OOS_Start":
                oos_df["Timestamp"].iloc[0],

            "OOS_End":
                oos_df["Timestamp"].iloc[-1],

            "Ending_Equity":
                ending_equity,

            "Net_OOS_Profit":
                stitched_net_profit,

            "Max_Drawdown_CNY":
                float(
                    oos_df["Drawdown"].min()
                ),

            "Positive_Windows":
                positive_windows,

            "Negative_Windows":
                negative_windows,

            "Flat_Windows":
                flat_windows,

            "Median_ChnLen":
                float(
                    optimization_df["ChnLen"]
                    .median()
                ),

            "Median_StpPct":
                float(
                    optimization_df["StpPct"]
                    .median()
                ),

            "StpPct_Lower_Bound_Share":
                lower_stop_share,

            "Accounting_Error":
                accounting_error,

            "Runtime_Seconds":
                spec_runtime,
        }
    )


    # --------------------------------------------------------
    # End-of-spec output
    # --------------------------------------------------------

    print()
    print(
        f"{label} COMPLETE"
    )

    print(
        f"OOS bars        : "
        f"{len(oos_df):,}"
    )

    print(
        f"OOS period      : "
        f"{oos_df['Timestamp'].iloc[0]} "
        f"to "
        f"{oos_df['Timestamp'].iloc[-1]}"
    )

    print(
        f"Net OOS profit  : "
        f"{stitched_net_profit:,.2f} CNY"
    )

    print(
        f"Ending equity   : "
        f"{ending_equity:,.2f} CNY"
    )

    print(
        f"Max drawdown    : "
        f"{float(oos_df['Drawdown'].min()):,.2f} CNY"
    )

    print(
        f"Accounting err. : "
        f"{accounting_error:.12f}"
    )

    print(
        f"Runtime         : "
        f"{spec_runtime:.2f} seconds"
    )


# ============================================================
# Final Section 5 summary
# ============================================================

sensitivity_run_summary = pd.DataFrame(
    spec_summary_rows
)


grand_runtime = (
    time.perf_counter()
    -
    grand_start_time
)


print()
print("=" * 110)
print("TASK M — FULL SENSITIVITY RUN SUMMARY")
print("=" * 110)

display(
    sensitivity_run_summary
)


print()
print(
    f"Total runtime: "
    f"{grand_runtime:.2f} seconds "
    f"({grand_runtime / 60.0:.2f} minutes)"
)


# ============================================================
# Task K baseline hard reconciliation
# ============================================================

baseline_oos = (
    SENSITIVITY_OOS_PATHS[
        "T4_tau3"
    ]
)

baseline_windows = (
    SENSITIVITY_WINDOW_RESULTS[
        "T4_tau3"
    ]
)


baseline_oos_bars_match = (
    len(baseline_oos)
    ==
    65_376
)

baseline_net_profit_match = np.isclose(
    baseline_oos["P&L"].sum(),
    855_221.26,
    rtol=0.0,
    atol=1e-6
)

baseline_ending_equity_match = np.isclose(
    baseline_oos["Equity"].iloc[-1],
    955_221.26,
    rtol=0.0,
    atol=1e-6
)

baseline_window_count_match = (
    len(baseline_windows)
    ==
    15
)


print()
print("=" * 88)
print("TASK K FULL BASELINE RECONCILIATION")
print("=" * 88)

print(
    f"OOS bars match         : "
    f"{baseline_oos_bars_match}"
)

print(
    f"Net OOS profit         : "
    f"{baseline_oos['P&L'].sum():,.6f}"
)

print(
    f"Net OOS profit match   : "
    f"{baseline_net_profit_match}"
)

print(
    f"Ending equity          : "
    f"{baseline_oos['Equity'].iloc[-1]:,.6f}"
)

print(
    f"Ending equity match    : "
    f"{baseline_ending_equity_match}"
)

print(
    f"OOS window count match : "
    f"{baseline_window_count_match}"
)


assert baseline_oos_bars_match
assert baseline_net_profit_match
assert baseline_ending_equity_match
assert baseline_window_count_match


print()
print(
    "ALL SIX TASK M SPECIFICATIONS COMPLETED, "
    "AND THE T4_tau3 BASELINE REPRODUCES TASK K."
)


SPECIFICATION 1/6 — T4_tau3
T = 4 years, tau = 3 months
OOS windows: 15
Status     : Task K baseline

[01/15] T4_tau3 | L=  500 | S=0.005 | IS Obj=   47.9633 | OOS P&L=   14,512.97 |   6.68s
[02/15] T4_tau3 | L= 3460 | S=0.005 | IS Obj=   47.8866 | OOS P&L=   10,751.14 |   6.27s
[03/15] T4_tau3 | L= 3460 | S=0.005 | IS Obj=   49.1445 | OOS P&L=   16,935.88 |   6.64s
[04/15] T4_tau3 | L=  580 | S=0.005 | IS Obj=   40.0061 | OOS P&L=    8,900.39 |   5.41s
[05/15] T4_tau3 | L=  640 | S=0.005 | IS Obj=   38.1663 | OOS P&L=   21,706.99 |   5.32s
[06/15] T4_tau3 | L=  640 | S=0.005 | IS Obj=   37.6659 | OOS P&L=   25,603.09 |   5.31s
[07/15] T4_tau3 | L=  640 | S=0.005 | IS Obj=   37.5103 | OOS P&L=   35,006.78 |   5.32s
[08/15] T4_tau3 | L= 2020 | S=0.005 | IS Obj=   36.3582 | OOS P&L=   32,255.21 |   5.29s
[09/15] T4_tau3 | L=  640 | S=0.005 | IS Obj=   40.0688 | OOS P&L=   17,905.03 |   5.31s
[10/15] T4_tau3 | L= 2020 | S=0.005 | IS Obj=   36.7747 | OOS P&L=   22,059.41 |   5.40s
[11/15]

,Specification,T_years,tau_months,Baseline,Windows,OOS_Bars,OOS_Start,OOS_End,Ending_Equity,Net_OOS_Profit,Max_Drawdown_CNY,Positive_Windows,Negative_Windows,Flat_Windows,Median_ChnLen,Median_StpPct,StpPct_Lower_Bound_Share,Accounting_Error,Runtime_Seconds
0,T4_tau3,4,3,True,15,65376,2022-07-01 09:05:00,2026-03-31 15:00:00,"955,221.260000","855,221.260000","-17,014.177500",15,0,0,"1,920.000000",0.005000,1.000000,0.000000,84.388325
1,T4_tau6,4,6,False,7,61344,2022-07-01 09:05:00,2025-12-31 15:00:00,"654,896.362500","554,896.362500","-17,014.177500",7,0,0,640.000000,0.005000,1.000000,0.000000,40.888051
2,T5_tau3,5,3,False,11,47880,2023-07-03 09:05:00,2026-03-31 15:00:00,"927,237.005000","827,237.005000","-17,014.177500",11,0,0,710.000000,0.005000,1.000000,0.000000,86.895768
3,T5_tau6,5,6,False,5,43848,2023-07-03 09:05:00,2025-12-31 15:00:00,"605,391.695000","505,391.695000","-17,014.177500",5,0,0,640.000000,0.005000,1.000000,0.000000,37.394458
4,T6_tau3,6,3,False,7,30528,2024-07-01 09:05:00,2026-03-31 15:00:00,"794,828.825000","694,828.825000","-17,014.177500",7,0,0,"1,920.000000",0.005000,1.000000,0.000000,68.495814
5,T6_tau6,6,6,False,3,26496,2024-07-01 09:05:00,2025-12-31 15:00:00,"476,719.050000","376,719.050000","-17,014.177500",3,0,0,"1,920.000000",0.005000,1.000000,0.000000,29.208928



Total runtime: 347.29 seconds (5.79 minutes)

TASK K FULL BASELINE RECONCILIATION
OOS bars match         : True
Net OOS profit         : 855,221.260000
Net OOS profit match   : True
Ending equity          : 955,221.260000
Ending equity match    : True
OOS window count match : True

ALL SIX TASK M SPECIFICATIONS COMPLETED, AND THE T4_tau3 BASELINE REPRODUCES TASK K.


## 6. Common-Period Robustness Comparison

The six sensitivity specifications do not cover identical OOS histories because longer in-sample windows begin later and 6-month evaluation horizons end at the last complete semiannual boundary.

Therefore, raw ending equity and cumulative OOS profit from Section 5 are not directly comparable across specifications.

To make the sensitivity comparison fair, all six OOS paths are restricted to their common calendar interval:

**2024-07-01 through 2025-12-31**

Each common-period path is then rebased to CNY 100,000.

The primary robustness comparison uses:

- total return
- calendar-time CAGR
- maximum drawdown percentage
- daily Sharpe ratio
- Calmar ratio

This common-period analysis is used to evaluate robustness across \(T/\tau\) specifications. It is not used to select a new ex-post optimal specification.

In [8]:
COMMON_START = pd.Timestamp(
    "2024-07-01 09:05:00"
)

COMMON_END = pd.Timestamp(
    "2025-12-31 15:00:00"
)

COMMON_INITIAL_EQUITY = E0


# ------------------------------------------------------------
# Performance helper
# ------------------------------------------------------------

def compute_common_period_metrics(
    common_df,
    initial_equity=100_000.0
):

    df = (
        common_df
        .copy()
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )

    pnl = df["P&L"].to_numpy(
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Rebased equity
    # --------------------------------------------------------

    equity = (
        initial_equity
        +
        np.cumsum(pnl)
    )

    df["Common_Equity"] = equity


    # --------------------------------------------------------
    # Total return
    # --------------------------------------------------------

    ending_equity = float(
        equity[-1]
    )

    net_profit = (
        ending_equity
        -
        initial_equity
    )

    total_return = (
        ending_equity
        /
        initial_equity
        -
        1.0
    )


    # --------------------------------------------------------
    # Calendar-time CAGR
    # --------------------------------------------------------

    start_ts = pd.Timestamp(
        df["Timestamp"].iloc[0]
    )

    end_ts = pd.Timestamp(
        df["Timestamp"].iloc[-1]
    )

    elapsed_days = (
        end_ts
        -
        start_ts
    ).total_seconds() / 86_400.0

    elapsed_years = (
        elapsed_days
        /
        365.25
    )

    if (
        elapsed_years > 0.0
        and
        ending_equity > 0.0
    ):

        cagr = (
            ending_equity
            /
            initial_equity
        ) ** (
            1.0
            /
            elapsed_years
        ) - 1.0

    else:

        cagr = np.nan


    # --------------------------------------------------------
    # Drawdown
    # --------------------------------------------------------

    equity_with_initial = np.concatenate(
        (
            np.array(
                [initial_equity],
                dtype=np.float64
            ),
            equity
        )
    )

    running_peak = np.maximum.accumulate(
        equity_with_initial
    )

    drawdown_amount = (
        equity_with_initial
        -
        running_peak
    )

    drawdown_pct = (
        equity_with_initial
        /
        running_peak
        -
        1.0
    )

    max_drawdown_cny = float(
        drawdown_amount.min()
    )

    max_drawdown_pct = float(
        drawdown_pct.min()
    )


    # --------------------------------------------------------
    # Daily Sharpe
    #
    # Same methodology used in the finalized AUG analysis:
    # end-of-day equity percentage returns, including the
    # initial capital as the base for the first daily return.
    # --------------------------------------------------------

    temp = pd.DataFrame(
        {
            "Timestamp":
                df["Timestamp"],

            "Equity":
                equity,
        }
    )

    temp["Date"] = (
        pd.to_datetime(
            temp["Timestamp"]
        )
        .dt.normalize()
    )

    daily_equity = (
        temp
        .groupby("Date")["Equity"]
        .last()
        .sort_index()
    )

    daily_equity_with_initial = pd.concat(
        [
            pd.Series(
                [initial_equity],
                index=[
                    daily_equity.index[0]
                    -
                    pd.Timedelta(days=1)
                ]
            ),
            daily_equity,
        ]
    )

    daily_returns = (
        daily_equity_with_initial
        .pct_change()
        .dropna()
    )

    if (
        len(daily_returns) > 1
        and
        daily_returns.std(ddof=1) > 0.0
    ):

        daily_sharpe = float(
            np.sqrt(252.0)
            *
            daily_returns.mean()
            /
            daily_returns.std(ddof=1)
        )

    else:

        daily_sharpe = np.nan


    # --------------------------------------------------------
    # Calmar
    # --------------------------------------------------------

    if max_drawdown_pct < 0.0:

        calmar = float(
            cagr
            /
            abs(max_drawdown_pct)
        )

    else:

        calmar = np.nan


    return {
        "Start":
            start_ts,

        "End":
            end_ts,

        "Bars":
            len(df),

        "Starting_Equity":
            initial_equity,

        "Ending_Equity":
            ending_equity,

        "Net_Profit":
            net_profit,

        "Total_Return":
            total_return,

        "CAGR":
            cagr,

        "Max_Drawdown_CNY":
            max_drawdown_cny,

        "Max_Drawdown_Pct":
            max_drawdown_pct,

        "Daily_Sharpe":
            daily_sharpe,

        "Calmar":
            calmar,
    }


# ------------------------------------------------------------
# Slice every specification to the identical common period
# ------------------------------------------------------------

COMMON_OOS_PATHS = {}

common_summary_rows = []

for spec in SENSITIVITY_SPECS:

    label = spec["label"]

    full_oos = (
        SENSITIVITY_OOS_PATHS[
            label
        ]
        .copy()
    )

    common = full_oos.loc[
        (
            full_oos["Timestamp"]
            >=
            COMMON_START
        )
        &
        (
            full_oos["Timestamp"]
            <=
            COMMON_END
        )
    ].copy()

    common = (
        common
        .sort_values("Timestamp")
        .reset_index(drop=True)
    )

    if len(common) == 0:
        raise RuntimeError(
            f"No common-period observations for {label}."
        )


    # --------------------------------------------------------
    # Rebase common-period equity
    # --------------------------------------------------------

    common["Common_Equity"] = (
        COMMON_INITIAL_EQUITY
        +
        common["P&L"].cumsum()
    )

    common_peak = (
        common["Common_Equity"]
        .cummax()
    )

    common_peak = np.maximum(
        common_peak,
        COMMON_INITIAL_EQUITY
    )

    common["Common_Drawdown"] = (
        common["Common_Equity"]
        -
        common_peak
    )


    # --------------------------------------------------------
    # Save common path
    # --------------------------------------------------------

    COMMON_OOS_PATHS[
        label
    ] = common


    # --------------------------------------------------------
    # Performance metrics
    # --------------------------------------------------------

    metrics = compute_common_period_metrics(
        common,
        initial_equity=COMMON_INITIAL_EQUITY
    )

    metrics.update(
        {
            "Specification":
                label,

            "T_years":
                spec["T_years"],

            "tau_months":
                spec["tau_months"],

            "Baseline":
                spec["baseline"],
        }
    )

    common_summary_rows.append(
        metrics
    )


# ------------------------------------------------------------
# Build comparison table
# ------------------------------------------------------------

common_period_summary = pd.DataFrame(
    common_summary_rows
)


column_order = [
    "Specification",
    "T_years",
    "tau_months",
    "Baseline",
    "Start",
    "End",
    "Bars",
    "Starting_Equity",
    "Ending_Equity",
    "Net_Profit",
    "Total_Return",
    "CAGR",
    "Max_Drawdown_CNY",
    "Max_Drawdown_Pct",
    "Daily_Sharpe",
    "Calmar",
]

common_period_summary = (
    common_period_summary[
        column_order
    ]
)


# ------------------------------------------------------------
# Common-period validation
# ------------------------------------------------------------

common_starts = (
    common_period_summary["Start"]
)

common_ends = (
    common_period_summary["End"]
)

common_bar_counts = (
    common_period_summary["Bars"]
)

same_start = bool(
    common_starts.nunique()
    ==
    1
)

same_end = bool(
    common_ends.nunique()
    ==
    1
)

same_bar_count = bool(
    common_bar_counts.nunique()
    ==
    1
)


# ------------------------------------------------------------
# Timestamp identity check
# ------------------------------------------------------------

baseline_timestamps = (
    COMMON_OOS_PATHS[
        "T4_tau3"
    ]["Timestamp"]
    .reset_index(drop=True)
)

timestamp_matches = {}

for label, common in COMMON_OOS_PATHS.items():

    current_timestamps = (
        common["Timestamp"]
        .reset_index(drop=True)
    )

    timestamp_matches[label] = bool(
        current_timestamps.equals(
            baseline_timestamps
        )
    )


# ------------------------------------------------------------
# Accounting validation
# ------------------------------------------------------------

accounting_checks = {}

for label, common in COMMON_OOS_PATHS.items():

    pnl_sum = float(
        common["P&L"].sum()
    )

    ending_equity = float(
        common["Common_Equity"]
        .iloc[-1]
    )

    accounting_error = (
        ending_equity
        -
        COMMON_INITIAL_EQUITY
        -
        pnl_sum
    )

    accounting_checks[label] = (
        accounting_error
    )

    assert np.isclose(
        accounting_error,
        0.0,
        rtol=0.0,
        atol=1e-6
    )


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 110)
print("TASK M — COMMON-PERIOD ROBUSTNESS COMPARISON")
print("=" * 110)

print(
    f"Common period : "
    f"{common_period_summary['Start'].iloc[0]} "
    f"to "
    f"{common_period_summary['End'].iloc[0]}"
)

print(
    f"Bars per spec : "
    f"{int(common_period_summary['Bars'].iloc[0]):,}"
)

print()

display(
    common_period_summary
)


# ------------------------------------------------------------
# Validation report
# ------------------------------------------------------------

print()
print("=" * 88)
print("COMMON-PERIOD VALIDATION")
print("=" * 88)

print(
    f"Same start timestamp   : "
    f"{same_start}"
)

print(
    f"Same end timestamp     : "
    f"{same_end}"
)

print(
    f"Same bar count         : "
    f"{same_bar_count}"
)

print()

for label in SENSITIVITY_OOS_PATHS.keys():

    print(
        f"{label:10s} "
        f"timestamps match: "
        f"{timestamp_matches[label]} | "
        f"accounting error: "
        f"{accounting_checks[label]:.12f}"
    )


# ------------------------------------------------------------
# Hard validation
# ------------------------------------------------------------

assert same_start
assert same_end
assert same_bar_count

assert all(
    timestamp_matches.values()
)

assert all(
    abs(x) < 1e-6
    for x in accounting_checks.values()
)


print()
print(
    "ALL SIX SPECIFICATIONS ARE ALIGNED "
    "ON AN IDENTICAL COMMON OOS PERIOD."
)

TASK M — COMMON-PERIOD ROBUSTNESS COMPARISON
Common period : 2024-07-01 09:05:00 to 2025-12-31 15:00:00
Bars per spec : 26,496



,Specification,T_years,tau_months,Baseline,Start,End,Bars,Starting_Equity,Ending_Equity,Net_Profit,Total_Return,CAGR,Max_Drawdown_CNY,Max_Drawdown_Pct,Daily_Sharpe,Calmar
0,T4_tau3,4,3,True,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","438,391.475000","338,391.475000",3.383915,1.676813,"-17,014.177500",-0.141416,3.979571,11.857294
1,T4_tau6,4,6,False,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","467,703.515000","367,703.515000",3.677035,1.794759,"-17,014.177500",-0.141416,4.247695,12.691329
2,T5_tau3,5,3,False,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","438,391.475000","338,391.475000",3.383915,1.676813,"-17,014.177500",-0.141416,3.979571,11.857294
3,T5_tau6,5,6,False,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","467,703.515000","367,703.515000",3.677035,1.794759,"-17,014.177500",-0.141416,4.247695,12.691329
4,T6_tau3,6,3,False,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","443,671.475000","343,671.475000",3.436715,1.698249,"-17,014.177500",-0.141416,4.025184,12.008872
5,T6_tau6,6,6,False,2024-07-01 09:05:00,2025-12-31 15:00:00,26496,"100,000.000000","476,719.050000","376,719.050000",3.767191,1.830535,"-17,014.177500",-0.141416,4.301115,12.944313



COMMON-PERIOD VALIDATION
Same start timestamp   : True
Same end timestamp     : True
Same bar count         : True

T4_tau3    timestamps match: True | accounting error: 0.000000000000
T4_tau6    timestamps match: True | accounting error: -0.000000000116
T5_tau3    timestamps match: True | accounting error: 0.000000000000
T5_tau6    timestamps match: True | accounting error: -0.000000000116
T6_tau3    timestamps match: True | accounting error: -0.000000000058
T6_tau6    timestamps match: True | accounting error: 0.000000000000

ALL SIX SPECIFICATIONS ARE ALIGNED ON AN IDENTICAL COMMON OOS PERIOD.


## 7. Parameter Stability and Boundary Diagnostics

Performance robustness does not necessarily imply parameter stability.

This section therefore examines how the in-sample optimal parameters change across the six walk-forward specifications.

Particular attention is given to:

- the distribution and variation of selected `ChnLen`;
- the number of distinct channel lengths selected;
- the frequency with which `ChnLen` reaches either grid boundary;
- the distribution of selected `StpPct`;
- the frequency with which `StpPct` reaches the lower or upper boundary of the professor-specified grid.

A parameter repeatedly selected at a search boundary does not invalidate the reported backtest, because the prescribed grid is retained exactly. However, it indicates that the unconstrained optimum may lie outside the tested parameter range and should therefore be reported as a limitation rather than interpreted as evidence of parameter stability.

In [9]:
parameter_summary_rows = []


for spec in SENSITIVITY_SPECS:

    label = spec["label"]

    opt = (
        SENSITIVITY_OPTIMIZATIONS[
            label
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # ChnLen diagnostics
    # --------------------------------------------------------

    chn_values = (
        opt["ChnLen"]
        .astype(int)
    )

    chn_min = int(
        chn_values.min()
    )

    chn_max = int(
        chn_values.max()
    )

    chn_mean = float(
        chn_values.mean()
    )

    chn_median = float(
        chn_values.median()
    )

    chn_unique = int(
        chn_values.nunique()
    )

    chn_lower_bound_share = float(
        np.mean(
            chn_values.to_numpy()
            ==
            CHNLEN_GRID[0]
        )
    )

    chn_upper_bound_share = float(
        np.mean(
            chn_values.to_numpy()
            ==
            CHNLEN_GRID[-1]
        )
    )

    if len(chn_values) > 1:

        chn_changes = int(
            np.sum(
                np.diff(
                    chn_values.to_numpy()
                )
                != 0
            )
        )

    else:

        chn_changes = 0


    # --------------------------------------------------------
    # StpPct diagnostics
    # --------------------------------------------------------

    stop_values = (
        opt["StpPct"]
        .astype(float)
    )

    stop_min = float(
        stop_values.min()
    )

    stop_max = float(
        stop_values.max()
    )

    stop_mean = float(
        stop_values.mean()
    )

    stop_median = float(
        stop_values.median()
    )

    stop_unique = int(
        stop_values.nunique()
    )

    stop_lower_bound_share = float(
        np.mean(
            np.isclose(
                stop_values.to_numpy(),
                STPPCT_GRID[0],
                rtol=0.0,
                atol=1e-12
            )
        )
    )

    stop_upper_bound_share = float(
        np.mean(
            np.isclose(
                stop_values.to_numpy(),
                STPPCT_GRID[-1],
                rtol=0.0,
                atol=1e-12
            )
        )
    )

    if len(stop_values) > 1:

        stop_changes = int(
            np.sum(
                ~np.isclose(
                    np.diff(
                        stop_values.to_numpy()
                    ),
                    0.0,
                    rtol=0.0,
                    atol=1e-12
                )
            )
        )

    else:

        stop_changes = 0


    # --------------------------------------------------------
    # Collect summary
    # --------------------------------------------------------

    parameter_summary_rows.append(
        {
            "Specification":
                label,

            "T_years":
                spec["T_years"],

            "tau_months":
                spec["tau_months"],

            "Windows":
                len(opt),

            "ChnLen_Min":
                chn_min,

            "ChnLen_Max":
                chn_max,

            "ChnLen_Mean":
                chn_mean,

            "ChnLen_Median":
                chn_median,

            "ChnLen_Unique":
                chn_unique,

            "ChnLen_Changes":
                chn_changes,

            "ChnLen_Lower_Bound_Share":
                chn_lower_bound_share,

            "ChnLen_Upper_Bound_Share":
                chn_upper_bound_share,

            "StpPct_Min":
                stop_min,

            "StpPct_Max":
                stop_max,

            "StpPct_Mean":
                stop_mean,

            "StpPct_Median":
                stop_median,

            "StpPct_Unique":
                stop_unique,

            "StpPct_Changes":
                stop_changes,

            "StpPct_Lower_Bound_Share":
                stop_lower_bound_share,

            "StpPct_Upper_Bound_Share":
                stop_upper_bound_share,
        }
    )


parameter_stability_summary = pd.DataFrame(
    parameter_summary_rows
)


# ------------------------------------------------------------
# Display full summary
# ------------------------------------------------------------

print("=" * 120)
print("TASK M — PARAMETER STABILITY SUMMARY")
print("=" * 120)

display(
    parameter_stability_summary
)


# ------------------------------------------------------------
# ChnLen selection frequencies
# ------------------------------------------------------------

print()
print("=" * 88)
print("SELECTED ChnLen FREQUENCIES")
print("=" * 88)

for spec in SENSITIVITY_SPECS:

    label = spec["label"]

    opt = (
        SENSITIVITY_OPTIMIZATIONS[
            label
        ]
    )

    frequencies = (
        opt["ChnLen"]
        .value_counts()
        .sort_values(
            ascending=False
        )
    )

    print()
    print(
        f"{label} "
        f"(T={spec['T_years']}, "
        f"tau={spec['tau_months']})"
    )

    print(
        frequencies.to_string()
    )


# ------------------------------------------------------------
# Boundary diagnostics
# ------------------------------------------------------------

all_stop_lower_boundary = bool(
    np.isclose(
        parameter_stability_summary[
            "StpPct_Lower_Bound_Share"
        ],
        1.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)

no_stop_upper_boundary = bool(
    np.isclose(
        parameter_stability_summary[
            "StpPct_Upper_Bound_Share"
        ],
        0.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)

no_chn_upper_boundary = bool(
    np.isclose(
        parameter_stability_summary[
            "ChnLen_Upper_Bound_Share"
        ],
        0.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)


print()
print("=" * 88)
print("PARAMETER BOUNDARY DIAGNOSTICS")
print("=" * 88)

print(
    f"StpPct = 0.005 in every optimization : "
    f"{all_stop_lower_boundary}"
)

print(
    f"No StpPct upper-bound selections      : "
    f"{no_stop_upper_boundary}"
)

print(
    f"No ChnLen upper-bound selections      : "
    f"{no_chn_upper_boundary}"
)


# ------------------------------------------------------------
# Hard grid validation
# ------------------------------------------------------------

for label, opt in SENSITIVITY_OPTIMIZATIONS.items():

    assert (
        opt["ChnLen"]
        .isin(CHNLEN_GRID)
        .all()
    )

    assert np.all(
        [
            np.any(
                np.isclose(
                    STPPCT_GRID,
                    x,
                    rtol=0.0,
                    atol=1e-12
                )
            )
            for x
            in opt["StpPct"]
        ]
    )


assert all_stop_lower_boundary
assert no_stop_upper_boundary
assert no_chn_upper_boundary


print()
print(
    "ALL TASK M PARAMETER-STABILITY "
    "AND GRID-BOUNDARY CHECKS PASSED."
)

TASK M — PARAMETER STABILITY SUMMARY


,Specification,T_years,tau_months,Windows,ChnLen_Min,ChnLen_Max,ChnLen_Mean,ChnLen_Median,ChnLen_Unique,ChnLen_Changes,ChnLen_Lower_Bound_Share,ChnLen_Upper_Bound_Share,StpPct_Min,StpPct_Max,StpPct_Mean,StpPct_Median,StpPct_Unique,StpPct_Changes,StpPct_Lower_Bound_Share,StpPct_Upper_Bound_Share
0,T4_tau3,4,3,15,500,3460,"1,554.666667","1,920.000000",8,9,0.066667,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000
1,T4_tau6,4,6,7,500,3460,"1,420.000000",640.000000,4,3,0.142857,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000
2,T5_tau3,5,3,11,640,2030,"1,267.272727",710.000000,5,4,0.000000,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000
3,T5_tau6,5,6,5,640,2030,"1,196.000000",640.000000,2,1,0.000000,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000
4,T6_tau3,6,3,7,640,1920,"1,564.285714","1,920.000000",3,2,0.000000,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000
5,T6_tau6,6,6,3,640,1920,"1,493.333333","1,920.000000",2,1,0.000000,0.000000,0.005000,0.005000,0.005000,0.005000,1,0,1.000000,0.000000



SELECTED ChnLen FREQUENCIES

T4_tau3 (T=4, tau=3)
ChnLen
640     4
2030    3
3460    2
2020    2
500     1
580     1
1920    1
710     1

T4_tau6 (T=4, tau=6)
ChnLen
640     3
2030    2
500     1
3460    1

T5_tau3 (T=5, tau=3)
ChnLen
640     5
2030    3
2020    1
1920    1
710     1

T5_tau6 (T=5, tau=6)
ChnLen
640     3
2030    2

T6_tau3 (T=6, tau=3)
ChnLen
1920    5
640     1
710     1

T6_tau6 (T=6, tau=6)
ChnLen
1920    2
640     1

PARAMETER BOUNDARY DIAGNOSTICS
StpPct = 0.005 in every optimization : True
No StpPct upper-bound selections      : True
No ChnLen upper-bound selections      : True

ALL TASK M PARAMETER-STABILITY AND GRID-BOUNDARY CHECKS PASSED.


## 8. Sensitivity Performance Ranges and Interpretation

The common-period analysis shows whether the AUG walk-forward results remain broadly similar under reasonable alternative choices of the in-sample window \(T\) and OOS horizon \(\tau\).

The interpretation focuses on ranges across the six specifications rather than identifying a single "best" configuration.

Performance robustness is assessed using common-period CAGR, Daily Sharpe, maximum drawdown percentage, and Calmar ratio. Parameter robustness is assessed separately through the distribution of selected `ChnLen` values and the frequency of grid-boundary solutions.

Because the available AUG history is relatively short, especially for the 6-year specifications, the sensitivity results should be interpreted as a historical robustness diagnostic within the provided sample rather than as strong statistical evidence of parameter invariance.

In [10]:
# ------------------------------------------------------------
# Common-period performance ranges
# ------------------------------------------------------------

cagr_min = float(
    common_period_summary["CAGR"].min()
)

cagr_max = float(
    common_period_summary["CAGR"].max()
)

sharpe_min = float(
    common_period_summary["Daily_Sharpe"].min()
)

sharpe_max = float(
    common_period_summary["Daily_Sharpe"].max()
)

mdd_min = float(
    common_period_summary["Max_Drawdown_Pct"].min()
)

mdd_max = float(
    common_period_summary["Max_Drawdown_Pct"].max()
)

calmar_min = float(
    common_period_summary["Calmar"].min()
)

calmar_max = float(
    common_period_summary["Calmar"].max()
)

ending_equity_min = float(
    common_period_summary["Ending_Equity"].min()
)

ending_equity_max = float(
    common_period_summary["Ending_Equity"].max()
)


# ------------------------------------------------------------
# Parameter-stability ranges
# ------------------------------------------------------------

chn_median_min = float(
    parameter_stability_summary[
        "ChnLen_Median"
    ].min()
)

chn_median_max = float(
    parameter_stability_summary[
        "ChnLen_Median"
    ].max()
)

chn_unique_min = int(
    parameter_stability_summary[
        "ChnLen_Unique"
    ].min()
)

chn_unique_max = int(
    parameter_stability_summary[
        "ChnLen_Unique"
    ].max()
)

stop_lower_share_min = float(
    parameter_stability_summary[
        "StpPct_Lower_Bound_Share"
    ].min()
)

stop_lower_share_max = float(
    parameter_stability_summary[
        "StpPct_Lower_Bound_Share"
    ].max()
)


# ------------------------------------------------------------
# Window-count range
# ------------------------------------------------------------

window_min = int(
    sensitivity_run_summary[
        "Windows"
    ].min()
)

window_max = int(
    sensitivity_run_summary[
        "Windows"
    ].max()
)


# ------------------------------------------------------------
# Compact robustness summary
# ------------------------------------------------------------

robustness_summary = pd.DataFrame(
    {
        "Metric": [
            "Common-Period CAGR",
            "Common-Period Daily Sharpe",
            "Common-Period Max Drawdown (%)",
            "Common-Period Calmar",
            "Common-Period Ending Equity",
            "Median ChnLen",
            "Unique ChnLen Values",
            "StpPct Lower-Bound Share",
            "OOS Windows per Specification",
        ],
        "Minimum": [
            cagr_min,
            sharpe_min,
            mdd_min,
            calmar_min,
            ending_equity_min,
            chn_median_min,
            chn_unique_min,
            stop_lower_share_min,
            window_min,
        ],
        "Maximum": [
            cagr_max,
            sharpe_max,
            mdd_max,
            calmar_max,
            ending_equity_max,
            chn_median_max,
            chn_unique_max,
            stop_lower_share_max,
            window_max,
        ],
    }
)


print("=" * 100)
print("TASK M — ROBUSTNESS RANGE SUMMARY")
print("=" * 100)

display(
    robustness_summary
)


# ------------------------------------------------------------
# Additional specification-level comparison
# ------------------------------------------------------------

performance_comparison = (
    common_period_summary[
        [
            "Specification",
            "T_years",
            "tau_months",
            "Ending_Equity",
            "CAGR",
            "Max_Drawdown_Pct",
            "Daily_Sharpe",
            "Calmar",
        ]
    ]
    .copy()
)


print()
print("=" * 100)
print("COMMON-PERIOD SPECIFICATION COMPARISON")
print("=" * 100)

display(
    performance_comparison
)


# ------------------------------------------------------------
# Descriptive range widths
# ------------------------------------------------------------

cagr_range_width = (
    cagr_max
    -
    cagr_min
)

sharpe_range_width = (
    sharpe_max
    -
    sharpe_min
)

mdd_range_width = (
    mdd_max
    -
    mdd_min
)

calmar_range_width = (
    calmar_max
    -
    calmar_min
)


print()
print("=" * 88)
print("RANGE WIDTHS")
print("=" * 88)

print(
    f"CAGR range width          : "
    f"{cagr_range_width:.6f}"
)

print(
    f"Daily Sharpe range width  : "
    f"{sharpe_range_width:.6f}"
)

print(
    f"Max drawdown range width  : "
    f"{mdd_range_width:.6f}"
)

print(
    f"Calmar range width        : "
    f"{calmar_range_width:.6f}"
)


# ------------------------------------------------------------
# Final diagnostic flags
# ------------------------------------------------------------

all_common_period_profitable = bool(
    (
        common_period_summary[
            "Net_Profit"
        ]
        >
        0.0
    ).all()
)

all_common_period_sharpe_positive = bool(
    (
        common_period_summary[
            "Daily_Sharpe"
        ]
        >
        0.0
    ).all()
)

all_stop_solutions_at_lower_bound = bool(
    np.isclose(
        parameter_stability_summary[
            "StpPct_Lower_Bound_Share"
        ],
        1.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)


print()
print("=" * 88)
print("ROBUSTNESS DIAGNOSTIC FLAGS")
print("=" * 88)

print(
    f"All common-period specifications profitable : "
    f"{all_common_period_profitable}"
)

print(
    f"All common-period Sharpe ratios positive     : "
    f"{all_common_period_sharpe_positive}"
)

print(
    f"All stop solutions at lower grid boundary    : "
    f"{all_stop_solutions_at_lower_bound}"
)


assert all_common_period_profitable
assert all_common_period_sharpe_positive
assert all_stop_solutions_at_lower_bound


print()
print(
    "TASK M SENSITIVITY PERFORMANCE "
    "AND PARAMETER DIAGNOSTICS SUMMARIZED."
)

TASK M — ROBUSTNESS RANGE SUMMARY


,Metric,Minimum,Maximum
0,Common-Period CAGR,1.676813,1.830535
1,Common-Period Daily Sharpe,3.979571,4.301115
2,Common-Period Max Drawdown (%),-0.141416,-0.141416
3,Common-Period Calmar,11.857294,12.944313
4,Common-Period Ending Equity,"438,391.475000","476,719.050000"
5,Median ChnLen,640.000000,"1,920.000000"
6,Unique ChnLen Values,2.000000,8.000000
7,StpPct Lower-Bound Share,1.000000,1.000000
8,OOS Windows per Specification,3.000000,15.000000



COMMON-PERIOD SPECIFICATION COMPARISON


,Specification,T_years,tau_months,Ending_Equity,CAGR,Max_Drawdown_Pct,Daily_Sharpe,Calmar
0,T4_tau3,4,3,"438,391.475000",1.676813,-0.141416,3.979571,11.857294
1,T4_tau6,4,6,"467,703.515000",1.794759,-0.141416,4.247695,12.691329
2,T5_tau3,5,3,"438,391.475000",1.676813,-0.141416,3.979571,11.857294
3,T5_tau6,5,6,"467,703.515000",1.794759,-0.141416,4.247695,12.691329
4,T6_tau3,6,3,"443,671.475000",1.698249,-0.141416,4.025184,12.008872
5,T6_tau6,6,6,"476,719.050000",1.830535,-0.141416,4.301115,12.944313



RANGE WIDTHS
CAGR range width          : 0.153722
Daily Sharpe range width  : 0.321544
Max drawdown range width  : 0.000000
Calmar range width        : 1.087018

ROBUSTNESS DIAGNOSTIC FLAGS
All common-period specifications profitable : True
All common-period Sharpe ratios positive     : True
All stop solutions at lower grid boundary    : True

TASK M SENSITIVITY PERFORMANCE AND PARAMETER DIAGNOSTICS SUMMARIZED.


## 9. Final Findings and Validation

The AUG walk-forward sensitivity analysis produces three main findings.

First, historical OOS performance remains strong across all six tested \(T/\tau\) specifications when evaluated over the identical common OOS period from July 2024 through December 2025. CAGR, Daily Sharpe, and Calmar remain within relatively narrow ranges, while maximum drawdown is identical across the six specifications. Within this common historical period, the main performance conclusion therefore does not depend strongly on the particular choice among the tested walk-forward window designs.

Second, the optimized channel horizon exhibits moderate specification sensitivity. Selected `ChnLen` values vary across windows and specifications, although longer in-sample windows tend to produce more concentrated selections. This indicates that the strategy adapts its breakout horizon as the estimation sample changes rather than relying on a single invariant channel length.

Third, `StpPct` is selected at the lower professor-specified grid boundary of `0.005` in every optimization window across every sensitivity specification. This persistent boundary solution is an important limitation. The prescribed parameter grid is retained unchanged for comparability with the original project, but the results should not be interpreted as evidence that `0.005` is an unconstrained interior optimum.

Overall, the sensitivity exercise supports the conclusion that the strong historical AUG walk-forward results are not driven solely by the original `(4-year IS, 3-month OOS)` window choice. However, this evidence is limited by the relatively short AUG history, the small number of OOS windows available for the longest specifications, the persistent stop-loss boundary solution, and the previously identified sensitivity of AUG P&L to session-gap attribution.

Accordingly, Task M is interpreted as a historical robustness diagnostic within the provided bar data rather than proof of deployable live performance, parameter invariance, or absence of overfitting and data-path bias.

In [11]:
# ------------------------------------------------------------
# Final headline metrics
# ------------------------------------------------------------

final_summary = pd.DataFrame(
    {
        "Item": [
            "Baseline Specification",
            "Sensitivity T Values",
            "Sensitivity Tau Values",
            "Number of Specifications",
            "Common OOS Start",
            "Common OOS End",
            "Common OOS Bars",
            "Common-Period CAGR Range",
            "Common-Period Daily Sharpe Range",
            "Common-Period MDD Range",
            "Common-Period Calmar Range",
            "Median ChnLen Range",
            "StpPct Lower-Bound Share",
            "Minimum OOS Windows",
            "Maximum OOS Windows",
        ],

        "Result": [
            "T4_tau3",
            "4, 5, 6 years",
            "3, 6 months",
            len(SENSITIVITY_SPECS),

            str(
                common_period_summary[
                    "Start"
                ].iloc[0]
            ),

            str(
                common_period_summary[
                    "End"
                ].iloc[0]
            ),

            f"{int(common_period_summary['Bars'].iloc[0]):,}",

            (
                f"{100 * cagr_min:.2f}% "
                f"to "
                f"{100 * cagr_max:.2f}%"
            ),

            (
                f"{sharpe_min:.4f} "
                f"to "
                f"{sharpe_max:.4f}"
            ),

            (
                f"{100 * mdd_min:.2f}% "
                f"to "
                f"{100 * mdd_max:.2f}%"
            ),

            (
                f"{calmar_min:.4f} "
                f"to "
                f"{calmar_max:.4f}"
            ),

            (
                f"{chn_median_min:.0f} "
                f"to "
                f"{chn_median_max:.0f}"
            ),

            (
                f"{100 * stop_lower_share_min:.2f}% "
                f"to "
                f"{100 * stop_lower_share_max:.2f}%"
            ),

            window_min,
            window_max,
        ],
    }
)


print("=" * 100)
print("TASK M — FINAL SUMMARY")
print("=" * 100)

display(
    final_summary
)


# ============================================================
# Final validation suite
# ============================================================

validation_results = {}


# ------------------------------------------------------------
# 1. Six specifications completed
# ------------------------------------------------------------

validation_results[
    "Six sensitivity specifications completed"
] = (
    len(SENSITIVITY_OOS_PATHS)
    ==
    6
)


# ------------------------------------------------------------
# 2. Task K baseline reproduced
# ------------------------------------------------------------

baseline_path = (
    SENSITIVITY_OOS_PATHS[
        "T4_tau3"
    ]
)

validation_results[
    "Task K baseline OOS bars reproduced"
] = (
    len(baseline_path)
    ==
    65_376
)

validation_results[
    "Task K baseline net profit reproduced"
] = bool(
    np.isclose(
        baseline_path["P&L"].sum(),
        855_221.26,
        rtol=0.0,
        atol=1e-6
    )
)

validation_results[
    "Task K baseline ending equity reproduced"
] = bool(
    np.isclose(
        baseline_path["Equity"].iloc[-1],
        955_221.26,
        rtol=0.0,
        atol=1e-6
    )
)


# ------------------------------------------------------------
# 3. Identical common-period alignment
# ------------------------------------------------------------

validation_results[
    "Common-period start identical"
] = (
    common_period_summary[
        "Start"
    ].nunique()
    ==
    1
)

validation_results[
    "Common-period end identical"
] = (
    common_period_summary[
        "End"
    ].nunique()
    ==
    1
)

validation_results[
    "Common-period bar count identical"
] = (
    common_period_summary[
        "Bars"
    ].nunique()
    ==
    1
)


# ------------------------------------------------------------
# 4. Common-period accounting
# ------------------------------------------------------------

common_accounting_pass = True

for label, common in COMMON_OOS_PATHS.items():

    pnl_sum = float(
        common["P&L"].sum()
    )

    ending_equity = float(
        common[
            "Common_Equity"
        ].iloc[-1]
    )

    error = (
        ending_equity
        -
        E0
        -
        pnl_sum
    )

    if not np.isclose(
        error,
        0.0,
        rtol=0.0,
        atol=1e-6
    ):
        common_accounting_pass = False


validation_results[
    "Common-period accounting reconciled"
] = common_accounting_pass


# ------------------------------------------------------------
# 5. Every specification profitable
# ------------------------------------------------------------

validation_results[
    "All common-period specifications profitable"
] = bool(
    (
        common_period_summary[
            "Net_Profit"
        ]
        >
        0.0
    ).all()
)


# ------------------------------------------------------------
# 6. Positive Sharpe across all specifications
# ------------------------------------------------------------

validation_results[
    "All common-period Sharpe ratios positive"
] = bool(
    (
        common_period_summary[
            "Daily_Sharpe"
        ]
        >
        0.0
    ).all()
)


# ------------------------------------------------------------
# 7. Parameter-grid integrity
# ------------------------------------------------------------

grid_integrity_pass = True

for label, opt in SENSITIVITY_OPTIMIZATIONS.items():

    chn_ok = (
        opt["ChnLen"]
        .isin(CHNLEN_GRID)
        .all()
    )

    stop_ok = np.all(
        [
            np.any(
                np.isclose(
                    STPPCT_GRID,
                    value,
                    rtol=0.0,
                    atol=1e-12
                )
            )
            for value
            in opt["StpPct"]
        ]
    )

    evaluations_ok = (
        (
            opt["Evaluations"]
            ==
            N_GRID_COMBINATIONS
        )
        .all()
    )

    if not (
        chn_ok
        and
        stop_ok
        and
        evaluations_ok
    ):
        grid_integrity_pass = False


validation_results[
    "Full parameter-grid integrity preserved"
] = grid_integrity_pass


# ------------------------------------------------------------
# 8. Stop boundary diagnostic
# ------------------------------------------------------------

validation_results[
    "StpPct lower-bound solution identified"
] = bool(
    np.isclose(
        parameter_stability_summary[
            "StpPct_Lower_Bound_Share"
        ],
        1.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)


# ------------------------------------------------------------
# 9. No ChnLen upper-bound solutions
# ------------------------------------------------------------

validation_results[
    "No ChnLen upper-bound solutions"
] = bool(
    np.isclose(
        parameter_stability_summary[
            "ChnLen_Upper_Bound_Share"
        ],
        0.0,
        rtol=0.0,
        atol=1e-12
    ).all()
)


# ------------------------------------------------------------
# Build validation table
# ------------------------------------------------------------

validation_table = pd.DataFrame(
    {
        "Validation Check":
            list(
                validation_results.keys()
            ),

        "Passed":
            list(
                validation_results.values()
            ),
    }
)


print()
print("=" * 100)
print("TASK M — FINAL VALIDATION")
print("=" * 100)

display(
    validation_table
)


# ------------------------------------------------------------
# Hard final assertion
# ------------------------------------------------------------

all_validations_passed = bool(
    validation_table[
        "Passed"
    ].all()
)

assert all_validations_passed


# ------------------------------------------------------------
# Final interpretation
# ------------------------------------------------------------

print()
print("=" * 100)
print("FINAL INTERPRETATION")
print("=" * 100)

print(
    "1. AUG historical OOS performance remains strong across "
    "all six tested T/tau specifications over the identical "
    "common OOS period."
)

print(
    "2. The common-period performance range is relatively "
    "contained, indicating limited sensitivity to the tested "
    "walk-forward window choices."
)

print(
    "3. ChnLen exhibits moderate variation across specifications, "
    "while StpPct is persistently selected at the 0.005 lower "
    "grid boundary."
)

print(
    "4. The stop-loss boundary solution, short AUG history, "
    "small number of windows for the longest specifications, "
    "and previously identified session-gap dependence remain "
    "material limitations."
)

print(
    "5. These results support historical robustness within the "
    "provided bar data, but do not establish deployable live "
    "performance or prove the absence of overfitting or "
    "data-path bias."
)

print()
print(
    "ALL TASK M FINAL VALIDATION CHECKS PASSED."
)

TASK M — FINAL SUMMARY


,Item,Result
0,Baseline Specification,T4_tau3
1,Sensitivity T Values,"4, 5, 6 years"
2,Sensitivity Tau Values,"3, 6 months"
3,Number of Specifications,6
4,Common OOS Start,2024-07-01 09:05:00
5,Common OOS End,2025-12-31 15:00:00
6,Common OOS Bars,"26,496"
7,Common-Period CAGR Range,167.68% to 183.05%
8,Common-Period Daily Sharpe Range,3.9796 to 4.3011
9,Common-Period MDD Range,-14.14% to -14.14%



TASK M — FINAL VALIDATION


,Validation Check,Passed
0,Six sensitivity specifications completed,True
1,Task K baseline OOS bars reproduced,True
2,Task K baseline net profit reproduced,True
3,Task K baseline ending equity reproduced,True
4,Common-period start identical,True
5,Common-period end identical,True
6,Common-period bar count identical,True
7,Common-period accounting reconciled,True
8,All common-period specifications profitable,True
9,All common-period Sharpe ratios positive,True



FINAL INTERPRETATION
1. AUG historical OOS performance remains strong across all six tested T/tau specifications over the identical common OOS period.
2. The common-period performance range is relatively contained, indicating limited sensitivity to the tested walk-forward window choices.
3. ChnLen exhibits moderate variation across specifications, while StpPct is persistently selected at the 0.005 lower grid boundary.
4. The stop-loss boundary solution, short AUG history, small number of windows for the longest specifications, and previously identified session-gap dependence remain material limitations.
5. These results support historical robustness within the provided bar data, but do not establish deployable live performance or prove the absence of overfitting or data-path bias.

ALL TASK M FINAL VALIDATION CHECKS PASSED.
